# J/ψ + D* efficiency-response systematic study

This is a **dedicated validation notebook**.  It is meant to be run before the final combined analysis notebook and its outputs are designed to be imported later.

It addresses three questions with the inputs that already exist:

1. **raw-map reconstruction closure** — do the central efficiency maps reproduce the stored weighted numerator/denominator sums exactly?
2. **Run-2 response-year sensitivity** — how much does the extracted $f_{\mathrm{DPS}}$ move if the detector response is taken as 2017-only or 2018-only instead of the luminosity-weighted 2017+2018 nominal response?
3. **efficiency-map binning stability** — how much does $f_{\mathrm{DPS}}$ move under exact coarsenings reconstructed from the stored raw weighted sums?

The notebook deliberately does **not** assign a fake independent-closure systematic.  A true independent closure requires a truth/test sample statistically independent of the map construction.  The output JSON records that item as pending rather than assigning an arbitrary number.


In [ ]:
from pathlib import Path
import os
import re
import math
import json
import gc
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    import uproot
except Exception as exc:
    raise RuntimeError("Install uproot before running this notebook.") from exc

try:
    from scipy.stats import beta as beta_distribution, chi2 as chi2_distribution, spearmanr
    from scipy.optimize import minimize_scalar
except Exception as exc:
    raise RuntimeError("Install scipy before running this notebook.") from exc

try:
    from iminuit import Minuit
except Exception as exc:
    raise RuntimeError(
        "This notebook uses iminuit for the profiled likelihood. Install iminuit first."
    ) from exc

try:
    import mplhep as hep
    HAS_MPLHEP = True
except Exception:
    HAS_MPLHEP = False

try:
    from tqdm.auto import tqdm
except Exception:
    tqdm = None


def configure_plot_style():
    if HAS_MPLHEP:
        plt.style.use(hep.style.CMS)
    plt.rcParams.update({
        "figure.dpi": 105,
        "savefig.dpi": 220,
        "axes.linewidth": 1.2,
        "legend.frameon": True,
        "legend.framealpha": 0.92,
        "legend.facecolor": "white",
        "legend.edgecolor": "0.75",
        "mathtext.fontset": "stixsans",
        "axes.formatter.use_mathtext": True,
        "axes.grid": False,
        "xtick.direction": "in",
        "ytick.direction": "in",
        "xtick.top": True,
        "ytick.right": True,
    })


def cms_label(ax, label="Internal", lumi_fb=None):
    if HAS_MPLHEP:
        try:
            hep.cms.label(label, data=True, lumi=lumi_fb, com=13, ax=ax, loc=0)
            return
        except Exception:
            pass
    ax.text(0.02, 1.07, "CMS", transform=ax.transAxes,
            ha="left", va="top", fontsize=22, fontweight="bold")
    ax.text(0.16, 1.07, label, transform=ax.transAxes,
            ha="left", va="top", fontsize=22, style="italic")
    if lumi_fb is not None:
        ax.text(0.98, 1.08, rf"{lumi_fb:.1f} fb$^{{-1}}$ (13 TeV)",
                transform=ax.transAxes, ha="right", va="top", fontsize=20)


def save_and_show(fig, stem, outdir=None):
    outdir = Path(outdir or OUTDIR)
    outdir.mkdir(parents=True, exist_ok=True)
    for extension in ("png", "pdf"):
        path = outdir / f"{stem}.{extension}"
        fig.savefig(path, bbox_inches="tight", facecolor="white")
        print("[plot] wrote", path)
    plt.show()
    plt.close(fig)


def safe_divide(num, den, default=0.0):
    num = np.asarray(num, dtype=float)
    den = np.asarray(den, dtype=float)
    out = np.full(np.broadcast(num, den).shape, default, dtype=float)
    return np.divide(num, den, out=out,
                     where=np.isfinite(den) & (den != 0.0))


def weighted_effective_entries(weights):
    weights = np.asarray(weights, dtype=float)
    weights = weights[np.isfinite(weights)]
    if weights.size == 0:
        return 0.0
    s1 = float(np.sum(weights))
    s2 = float(np.sum(weights**2))
    return float(s1*s1/s2) if s2 > 0.0 else 0.0


def nearest_psd(matrix, relative_floor=1.0e-12):
    matrix = np.asarray(matrix, dtype=float)
    matrix = 0.5*(matrix + matrix.T)
    eigval, eigvec = np.linalg.eigh(matrix)
    scale = max(float(np.max(np.abs(eigval))), 1.0e-20)
    floor = relative_floor*scale
    eigval = np.maximum(eigval, floor)
    return eigvec @ np.diag(eigval) @ eigvec.T


def rms_covariance_about_nominal(samples, nominal):
    """Second-moment covariance around a nominal state; useful for approved variations."""
    arr = np.asarray(samples, dtype=float)
    nominal = np.asarray(nominal, dtype=float)
    if arr.ndim != 2 or len(arr) == 0:
        return np.zeros((nominal.size, nominal.size), dtype=float)
    delta = arr - nominal[None, :]
    return 0.5*((delta.T @ delta)/float(len(delta)) + ((delta.T @ delta)/float(len(delta))).T)


def sample_covariance_centered(samples, n_dim=None):
    """Ordinary sample covariance around the ensemble mean."""
    arr = np.asarray(samples, dtype=float)
    if arr.ndim != 2 or len(arr) < 2:
        n = int(n_dim or (arr.shape[1] if arr.ndim == 2 else 1))
        return np.zeros((n, n), dtype=float)
    cov = np.cov(arr, rowvar=False, ddof=1)
    cov = np.atleast_2d(np.asarray(cov, dtype=float))
    return 0.5*(cov+cov.T)


configure_plot_style()
print("uproot:", uproot.__version__)
print("iminuit imported")
print("mplhep available:", HAS_MPLHEP)


In [ ]:
# ================================================================
# PRIMARY ANALYSIS FLAGS
# ================================================================
# Nominal choice: one physics parameter, with the ccbar/bbbar composition
# fixed to the central fiducial MC prediction.
#
# False -> only fDPS is free; ccbar/bbbar fractions are fixed from the
#          central fiducial MC composition (1 free parameter).
# True -> fDPS, f_cc_SPS and f_cc_DPS are fitted.
FREE_CC_FRACTIONS = False

# Optional Gaussian constraints on the ccbar/bbbar compositions.
# The central values are obtained automatically from the same fiducial MC
# composition used in the 1-parameter fit.
USE_CC_FRACTION_CONSTRAINTS = True

# Absolute uncertainty on the fractions.
# Exploratory values for now; must be physically justified for the nominal result.
CC_SPS_CONSTRAINT_SIGMA = 0.02
CC_DPS_CONSTRAINT_SIGMA = 0.02

# False -> fit |Delta phi| in [0, 3.0], as in the current FINAL notebook.
# True  -> fit the full folded range [0, pi].
USE_DPHI_PI = True

# False -> 3x3 fit.
# True  -> use the legacy '6x6' key.  Its explicit edge arrays currently
#          define 4x4=16 bins; the key is retained to avoid a broad rewrite.
USE_6X6_BINNING = True

# False -> do not add the integrated AN-21-143 systematic budget to the
#          absolute-spectrum data uncertainty.
# True  -> include it as a fully correlated normalization uncertainty.
#
# IMPORTANT: this is a shape-only fDPS likelihood. A fully correlated global
# normalization uncertainty cancels from the normalized fit and from Data/MC.
# Therefore this flag changes the absolute-spectrum uncertainty (and bookkeeping),
# but it does not artificially broaden the fDPS profile. Only genuine differential
# variations can change the shape likelihood.
INCLUDE_AN143_INTEGRATED_SYSTEMATICS = True

# Optional: run the non-nominal binning in the same notebook as an audit cross-check.
RUN_ALTERNATE_BINNING_CROSSCHECK = False

# Data-driven DPS-enriched normalization region.
#
# True:
#   scan the allowed (|Delta y|, |Delta phi|) rectangles and select the
#   maximum-purity region after the predeclared support/closure gates.
#
# False:
#   use the fixed comparison region
#       |Delta y| >= 1.8
#       pi/3 <= |Delta phi| <= 2pi/3
#
# This choice affects only the data-driven cross-check.  It does not change
# the nominal 2D profile-likelihood fit.
AUTO_DD_REGION = True

# Apply the DPS purity from the template fit as a scale factor
# to the data-driven extraction.
#
# True:
#   fDPS_DD = f_raw_DD * P_DPS(post-fit)
#
# False:
#   fDPS_DD = f_raw_DD
#
# The post-fit purity is still calculated and kept as a diagnostic,
# but it is not applied to the data-driven result when False.
APPLY_DD_PURITY_SCALE = True


# The available MC does not describe the normalized 2D shape within its known
# finite-sample uncertainties.
USE_MODEL_INADEQUACY = True

MODEL_INADEQUACY_REL = float(os.environ.get("JPSIDSTAR_MODEL_INADEQUACY_REL", "0.25"))
MODEL_INADEQUACY_SOURCE = (
    "single global K=I relative 2D shape floor, calibrated on nominal residuals"
)
MODEL_INADEQUACY_CALIBRATED_ON_SAME_DATA = True
MODEL_INADEQUACY_EXTERNALLY_VALIDATED = False
if not (0.0 <= MODEL_INADEQUACY_REL <= 0.50):
    raise ValueError("MODEL_INADEQUACY_REL must be between 0 and 0.50")

# ================================================================
# Paths
# ================================================================
PROJECT_DIR = Path(os.environ.get(
    "JPSIDSTAR_PROJECT_DIR",
    "/eos/home-j/jgomespi/SWAN_projects/Associated_Production_JPsi_Dstar",
)).expanduser()
DATA_DIR = Path(
    os.environ.get("JPSIDSTAR_DATA_DIR", str(PROJECT_DIR / "Template_fit"))
).expanduser()
EFFICIENCY_DIR = Path(
    os.environ.get(
        "JPSIDSTAR_EFF_DIR",
        "/afs/cern.ch/work/j/jgomespi/private/workspace/JPsiDstar/"
        "Efficiency_Quarkonia_OpenCharm/output/efficiency/final",
    )
).expanduser()

DPHI_MAX = float(np.pi if USE_DPHI_PI else 3.0)
NOMINAL_BINNING = "6x6" if USE_6X6_BINNING else "3x3"
FIT_PARAMETER_MODE = "3par" if FREE_CC_FRACTIONS else "1par"
DPHI_RANGE_TAG = "pi" if USE_DPHI_PI else "3p0"
AN_TAG = "ANon" if INCLUDE_AN143_INTEGRATED_SYSTEMATICS else "ANoff"
FLOOR_TAG = (
    f"MCfloor{int(round(100*MODEL_INADEQUACY_REL)):02d}"
    if USE_MODEL_INADEQUACY else "MCfloorOff"
)

DD_REGION_TAG = "DDauto" if AUTO_DD_REGION else "DDfixed"
DD_SCALE_TAG = "DDscaleOn" if APPLY_DD_PURITY_SCALE else "DDscaleOff"

DATA_PERIOD = "Run2"
EFFICIENCY_SCOPE_TAG = "Eff1718"
RUN_SCOPE_TAG = f"{DATA_PERIOD}Data_{EFFICIENCY_SCOPE_TAG}"

CONFIG_TAG = (
    f"{RUN_SCOPE_TAG}_{FIT_PARAMETER_MODE}_{NOMINAL_BINNING}_"
    f"dphi{DPHI_RANGE_TAG}_{AN_TAG}_{FLOOR_TAG}_"
    f"{DD_REGION_TAG}_{DD_SCALE_TAG}"
)

OUTDIR = Path(os.environ["JPSIDSTAR_SYSTEMATICS_OUTPUT_DIR"]).expanduser()
OUTDIR.mkdir(parents=True, exist_ok=True)

DATA_FILE = "Charmonium_all_vtx0p05_sigma_eff_jpsi_pt_bin1_25_100.root"
SWEIGHT_FILE = (
    "Charmonium_all_vtx0p05_sigma_eff_jpsi_pt_bin1_25_100_"
    "3Dfit_splot_signal_weights.root"
)
MC_TREE_WEIGHT_BRANCH = "weight"

# Frozen entry-count contract for the historical full-Run-2 data/sPlot pair.
REFERENCE_RAW_RUN2_ENTRIES = 36208
REFERENCE_SPLOT_ENTRIES = 25554

# ================================================================
# Physics normalization / fiducial phase space
# ================================================================
LUMI_RUN2_FB_INV = 126.28
LUMI_RUN2_PB_INV = 1000.0 * LUMI_RUN2_FB_INV
BR_JPSI_TO_MUMU = 0.05961
BR_DSTAR_TO_D0PI = 0.677
BR_D0_TO_KPI = 0.03947
BRANCHING_PRODUCT = BR_JPSI_TO_MUMU * BR_DSTAR_TO_D0PI * BR_D0_TO_KPI

JPSI_PT_MIN, JPSI_PT_MAX = 25.0, 100.0
JPSI_ABSY_MAX = 1.2
JPSI_MASS_MIN, JPSI_MASS_MAX = 2.95, 3.25
JPSI_DL_MIN, JPSI_DL_MAX = -0.5, 1.4
DSTAR_PT_MIN, DSTAR_PT_MAX = 4.0, 60.0
DSTAR_ABSY_MAX = 2.1

# ================================================================
# 2D binnings
# ================================================================
BINNING = {
    "3x3": {
        "dphi": np.array([
            0.0,
            DPHI_MAX / 3.0,
            2.0 * DPHI_MAX / 3.0,
            DPHI_MAX,
        ]),
        "dy": np.array([
            0.0,
            0.9,
            1.8,
            3.6,
        ]),
    },

    "6x6": {
        "dphi": np.array([
            0.0,
            DPHI_MAX / 4.0,
            2.0 * DPHI_MAX / 4.0,
            3.0 * DPHI_MAX / 4.0,
            DPHI_MAX,
        ]),
        "dy": np.array([
            0.0,
            0.733333333333333333333333,#0.75
            1.466666666666666666666666, # 1.5
            2.2, # 2.25
            4.0, #
        ]),
    },
}
ALTERNATE_BINNING = "3x3" if NOMINAL_BINNING == "6x6" else "6x6"
ACTUAL_N_DPHI_BINS = len(BINNING[NOMINAL_BINNING]["dphi"]) - 1
ACTUAL_N_DY_BINS = len(BINNING[NOMINAL_BINNING]["dy"]) - 1
MIN_TEMPLATE_NEFF_REPORTABLE = 10.0

# ================================================================
# Run-2 data period and currently available efficiency response
# ================================================================
# The data and sWeights are the historical FULL Run-2 inputs.
# They are intentionally kept intact.  The validated efficiency campaign
# currently contains 2017 and 2018; 2016APV/2016 will be incorporated later.
LUMI_BY_YEAR_FB = {
    "2016APV": 16.52,
    "2016": 8.53,
    "2017": 41.48,
    "2018": 59.75,
}

EFFICIENCY_YEARS = ("2017", "2018")
EFFICIENCY_YEAR_WEIGHTS = {
    year: float(LUMI_BY_YEAR_FB[year])
    for year in EFFICIENCY_YEARS
}
LUMI_EFFICIENCY_RESPONSE_FB_INV = float(
    sum(EFFICIENCY_YEAR_WEIGHTS.values())
)

# This flag is a scope label, not an event-level selection:
# full Run-2 data are corrected with the luminosity-weighted 2017+2018
# detector response until the 2016 maps are available.
USE_PARTIAL_RUN2_EFFICIENCY_PROXY = True
USE_2017_MC_AS_RUN2_PROXY = False  # compatibility/audit field only

EFFICIENCY_FLOOR = 1.0e-6
ASSOCIATION_EFF_MODE = "pt"

# Exactly one association representation is used in the nominal product.
EFFICIENCY_FACTORS_USED = [
    "acc_dimu",
    "acc_dstar",
    "eff_cuts_dimu",
    "eff_cuts_dstar",
    "eff_trigger",
    "eff_asso_pt",
]
EFFICIENCY_REQUIRED = EFFICIENCY_FACTORS_USED + ["eff_asso_rap"]
EFFICIENCY_SOURCE_FOR_COMPONENT = {
    "DPS-ccbar": "DPS-ccbar",
    "DPS-bbbar": "DPS-bbbar",
    "SPS-ccbar": "SPS-ccbar",
    "SPS-bbbar": "SPS-bbbar",
}

# Final efficiency ROOTs were validated with N_eff >= 25 for every map/bin
# entering the nominal product.
MIN_EFFICIENCY_MAP_NEFF = 25.0

# The current ROOT format stores per-factor/bin statistics but not the
# cross-factor event-level covariance. Independent factor fluctuations are
# therefore an explicit approximation, not a hidden exact statement.
EFFICIENCY_CROSS_FACTOR_COVARIANCE_AVAILABLE = False
# ================================================================
# Statistical ensemble / fit controls
# ================================================================
N_MODEL_STAT_REPLICAS = int(os.environ.get("JPSIDSTAR_MODEL_STAT_REPLICAS", "500"))
N_SOURCE_AUDIT_REPLICAS = int(os.environ.get("JPSIDSTAR_SOURCE_AUDIT_REPLICAS", "1200"))
N_PREDICTION_DATA_TOYS = int(os.environ.get("JPSIDSTAR_PREDICTION_DATA_TOYS", "500"))
N_PREDICTION_MODEL_REPLICAS = int(os.environ.get("JPSIDSTAR_PREDICTION_MODEL_REPLICAS", str(N_MODEL_STAT_REPLICAS)))
N_DD_DATA_REPLICAS = int(os.environ.get("JPSIDSTAR_DD_DATA_REPLICAS", "500"))
N_DD_MODEL_REPLICAS = int(os.environ.get("JPSIDSTAR_DD_MODEL_REPLICAS", "250"))
MIN_DD_SUCCESSFUL_REPLICAS = 100
MIN_SUCCESSFUL_REPLICAS = 100
MIN_MODEL_REPLICA_SUCCESS_FRACTION = 0.90
MIN_DATA_REPLICA_SUCCESS_FRACTION = 0.95
MAX_RELEVANT_SIGMA_MCSE_FRACTION = 0.05
N_PROFILE_SCAN = 81
N_GOF_TOYS = int(os.environ.get("JPSIDSTAR_PROFILE_GOF_TOYS", "1000"))
RANDOM_SEED = 12345
GOF_PVALUE_MIN_REPORTABLE = 0.01

# Explicit validation flag. Set only after a reviewed closure test.
SPLOT_CLOSURE_VALIDATED = bool(int(os.environ.get("JPSIDSTAR_SPLOT_CLOSURE_VALIDATED", "1")))

# ================================================================
# Differential systematic inputs
# ================================================================
SHAPE_SYSTEMATICS_DIR = Path(
    os.environ.get(
        "JPSIDSTAR_SHAPE_SYST_DIR",
        str(DATA_DIR / "Data" / "shape_systematics"),
    )
).expanduser()

# Signal-extraction/sPlot variants can be placed as ROOT files here.
SPLOT_VARIATION_DIR = SHAPE_SYSTEMATICS_DIR / "signal_yield_fit_variants"

# ================================================================
# Data-driven DPS-enriched region and mixing
# ================================================================
# AUTO_DD_REGION is defined above with the primary analysis flags.

# The DD method is always normalized in the explicitly clipped domain,
# even if a future template-fit cross-check uses the interval up to pi.
#DD_DPHI_MAX = min(3.0, DPHI_MAX)
DD_DPHI_MAX = DPHI_MAX

# Scan used only when AUTO_DD_REGION = True.
DD_SCAN_DY_MIN = np.arange(1.2, 3.01, 0.2, dtype=float)
DD_SCAN_DPHI_EDGES = np.arange(0.0, DD_DPHI_MAX + 0.001, 0.5, dtype=float)
DD_SCAN_MIN_DPHI_WIDTH = 1.0
DD_PURITY_PLATEAU = 0.01
DD_MIN_DPS_ACCEPTANCE = 0.02
DD_MAX_MIXED_DPS_CLOSURE_DEVIATION = 0.20
MIN_DATA_REGION_NEFF = 50.0
MIN_MIXED_REGION_NEFF = 1000.0
MIN_SPS_REGION_NEFF = 50.0
MIN_DPS_MC_REGION_NEFF = 50.0

# Fixed region used when AUTO_DD_REGION = False.
FIXED_DPS_REGION = {
    "dy_min": 2.2,
    "dphi_min": 0.0,
    "dphi_max": 4*DD_DPHI_MAX/5.0,
}

# In automatic mode this is only the initial seed and is overwritten
# after the nominal fit.  In fixed mode it remains the final DD region.
CENTRAL_DPS_REGION = FIXED_DPS_REGION.copy()

MIXING_INPUT_DIR = PROJECT_DIR / "condor_data_preparation" / "output_noHLT_preassoc_v1"
MIXING_INPUT_GLOB = "Run2/*/*/*/output_mixing_*.coffea"
REQUIRE_DIFFERENT_EVENT_FOR_MIXING = True
MAX_JPSI_MIX_POOL = int(os.environ.get("JPSIDSTAR_MAX_JPSI_POOL", "750000"))
MAX_DSTAR_MIX_POOL = int(os.environ.get("JPSIDSTAR_MAX_DSTAR_POOL", "250000"))
TARGET_MIXED_PAIRS = int(os.environ.get("JPSIDSTAR_TARGET_MIXED", "1000000"))
MIXING_RNG_SEED = 12345
MIX_CHUNK = int(os.environ.get("JPSIDSTAR_MIX_EFF_CHUNK", "100000"))

# ================================================================
# AN2021-143 v6 integrated uncertainty budget
# ================================================================
# These are integrated cross-section uncertainties. They are retained only
# as a fully correlated absolute-normalization reference unless an approved
# differential variation exists.
AN143_REFERENCE_RELATIVE_SYSTEMATICS = {
    "luminosity": 0.0242,
    "branching_ratio": 0.0205,
    "signal_yield": 0.0100,
    "tracking": 0.0925,
    "mc_sample_statistics": 0.0010,
    "pileup_reweighting": 0.1633,
    "yield_instability_pileup": 0.2000,
    "trigger_efficiency_model": 0.0340,
    "jpsi_polarization": 0.0,
}
AN143_TABLE_TOTAL_REL = 0.2784
AN143_REFERENCE_QUADRATURE_REL = math.sqrt(
    sum(v*v for v in AN143_REFERENCE_RELATIVE_SYSTEMATICS.values())
)
AN143_COMMON_NORMALIZATION_SOURCES = tuple(
    name for name in AN143_REFERENCE_RELATIVE_SYSTEMATICS
    if name != "mc_sample_statistics"
)
AN143_COMMON_REL = math.sqrt(sum(
    AN143_REFERENCE_RELATIVE_SYSTEMATICS[name]**2
    for name in AN143_COMMON_NORMALIZATION_SOURCES
))

print("AN143 table total             =", f"{100*AN143_TABLE_TOTAL_REL:.3f}%")
print("AN143 quadrature cross-check =", f"{100*AN143_REFERENCE_QUADRATURE_REL:.3f}%")
print("AN143 upper normalization ref =", f"{100*AN143_COMMON_REL:.3f}%")
print("model-stat replicas           =", N_MODEL_STAT_REPLICAS)
print("prediction data toys          =", N_PREDICTION_DATA_TOYS)
print("prediction model replicas     =", N_PREDICTION_MODEL_REPLICAS)
print("data-driven data replicas     =", N_DD_DATA_REPLICAS)
print("data-driven model replicas    =", N_DD_MODEL_REPLICAS)
print("GOF toys                      =", N_GOF_TOYS)
print("fit parameter mode             =", FIT_PARAMETER_MODE)
print("|Delta phi| maximum            =", DPHI_MAX)
print("nominal binning                =", NOMINAL_BINNING)
print("actual nominal grid            =", f"{ACTUAL_N_DPHI_BINS}x{ACTUAL_N_DY_BINS}")
print("alternate-binning cross-check  =", RUN_ALTERNATE_BINNING_CROSSCHECK)
print("data-driven region mode        =", "automatic maximum-purity" if AUTO_DD_REGION else "fixed")
print("AN143 integrated syst enabled  =", INCLUDE_AN143_INTEGRATED_SYSTEMATICS)
print("empirical MC-shape floor       =", f"{100*MODEL_INADEQUACY_REL:.1f}%" if USE_MODEL_INADEQUACY else "OFF")
print("sPlot closure validated       =", SPLOT_CLOSURE_VALIDATED)
print("data-driven purity scale       =", APPLY_DD_PURITY_SCALE)
print("data period                    =", DATA_PERIOD)
print("full Run-2 luminosity          =", f"{LUMI_RUN2_FB_INV:.2f} fb^-1")
print("efficiency-response years      =", list(EFFICIENCY_YEARS))
print("efficiency-response luminosity =", f"{LUMI_EFFICIENCY_RESPONSE_FB_INV:.2f} fb^-1")
print("2016 efficiency pending        =", USE_PARTIAL_RUN2_EFFICIENCY_PROXY)
print("OUTDIR:", OUTDIR.resolve())

ANALYSIS_CONFIGURATION = {
    "FREE_CC_FRACTIONS": bool(FREE_CC_FRACTIONS),
    "N_FREE_PARAMETERS": int(3 if FREE_CC_FRACTIONS else 1),
    "USE_DPHI_PI": bool(USE_DPHI_PI),
    "DPHI_MAX": float(DPHI_MAX),
    "USE_6X6_BINNING": bool(USE_6X6_BINNING),
    "NOMINAL_BINNING": NOMINAL_BINNING,
    "ACTUAL_N_DPHI_BINS": int(ACTUAL_N_DPHI_BINS),
    "ACTUAL_N_DY_BINS": int(ACTUAL_N_DY_BINS),
    "INCLUDE_AN143_INTEGRATED_SYSTEMATICS": bool(INCLUDE_AN143_INTEGRATED_SYSTEMATICS),
    "RUN_ALTERNATE_BINNING_CROSSCHECK": bool(RUN_ALTERNATE_BINNING_CROSSCHECK),
    "USE_MODEL_INADEQUACY": bool(USE_MODEL_INADEQUACY),
    "MODEL_INADEQUACY_REL": float(MODEL_INADEQUACY_REL if USE_MODEL_INADEQUACY else 0.0),
    "MODEL_INADEQUACY_SOURCE": MODEL_INADEQUACY_SOURCE,
    "MODEL_INADEQUACY_CALIBRATED_ON_SAME_DATA": bool(MODEL_INADEQUACY_CALIBRATED_ON_SAME_DATA),
    "MODEL_INADEQUACY_EXTERNALLY_VALIDATED": bool(MODEL_INADEQUACY_EXTERNALLY_VALIDATED),
    "N_MODEL_STAT_REPLICAS": int(N_MODEL_STAT_REPLICAS),
    "N_PREDICTION_DATA_TOYS": int(N_PREDICTION_DATA_TOYS),
    "N_PREDICTION_MODEL_REPLICAS": int(N_PREDICTION_MODEL_REPLICAS),
    "N_DD_DATA_REPLICAS": int(N_DD_DATA_REPLICAS),
    "N_DD_MODEL_REPLICAS": int(N_DD_MODEL_REPLICAS),
    "N_GOF_TOYS": int(N_GOF_TOYS),
    "DD_MAX_MIXED_DPS_CLOSURE_DEVIATION": float(DD_MAX_MIXED_DPS_CLOSURE_DEVIATION),
    "CONFIG_TAG": CONFIG_TAG,
    "AUTO_DD_REGION": bool(AUTO_DD_REGION),
    "DD_REGION_MODE": "automatic_maximum_purity" if AUTO_DD_REGION else "fixed",
    "FIXED_DPS_REGION": {
        key: float(value) for key, value in FIXED_DPS_REGION.items()
    },
    "APPLY_DD_PURITY_SCALE": bool(APPLY_DD_PURITY_SCALE),
    "DATA_PERIOD": DATA_PERIOD,
    "LUMI_RUN2_FB_INV": float(LUMI_RUN2_FB_INV),
    "EFFICIENCY_YEARS": list(EFFICIENCY_YEARS),
    "LUMI_EFFICIENCY_RESPONSE_FB_INV": float(LUMI_EFFICIENCY_RESPONSE_FB_INV),
    "USE_PARTIAL_RUN2_EFFICIENCY_PROXY": bool(USE_PARTIAL_RUN2_EFFICIENCY_PROXY),
    "ASSOCIATION_EFF_MODE": ASSOCIATION_EFF_MODE,
}
with open(OUTDIR/"analysis_configuration.json", "w") as handle:
    json.dump(ANALYSIS_CONFIGURATION, handle, indent=2)


In [ ]:
# ================================================================
# DEDICATED EFFICIENCY-SYSTEMATICS STUDY
# ================================================================
# This notebook is intentionally separate from the final physics notebook.
# It measures detector-response sensitivity terms and writes machine-readable
# outputs that can later be imported into the final combined notebook.
#
# It does NOT invent an independent closure uncertainty.  With the currently
# staged final ROOTs we can test exact raw-sum reconstruction and binning/year
# stability.  A true independent efficiency closure needs an independent
# truth/test sample (or a predeclared split of the efficiency production).

SYSTEMATICS_TAG = "EfficiencyYearBinning_20260904"
OUTDIR = Path(os.environ["JPSIDSTAR_SYSTEMATICS_OUTPUT_DIR"]).expanduser()
OUTDIR.mkdir(parents=True, exist_ok=True)

# Use the staged EOS copies explicitly; this avoids AFS visibility differences
# between lxplus and SWAN/Jupyter.
EFFICIENCY_DIR = Path(os.environ["JPSIDSTAR_EFF_DIR"]).expanduser()

# Only the nominal fit binning is needed in this study.
RUN_ALTERNATE_BINNING_CROSSCHECK = False

print("Systematics output directory:", OUTDIR)
print("Efficiency directory:", EFFICIENCY_DIR)


## Input preflight

In [ ]:
# ================================================================
# INPUT PREFLIGHT
# ================================================================
#
# Contract:
#
#   data:
#       full Run-2 data + full Run-2 sWeights
#
#   MC:
#       the eight selected-MC ROOT files already used by the analysis
#
#   efficiencies:
#       final validated 2017 + 2018 ROOT files
#
#   mixing:
#       full Run-2 coffea pool
#
# The final efficiency ROOT files are preferentially read from EOS.
# An AFS location is retained only as a secondary fallback.
# ================================================================

from pathlib import Path
import warnings


# ----------------------------------------------------------------
# Main analysis inputs
# ----------------------------------------------------------------

data_root_dir = DATA_DIR / "Data"

required_data_files = [
    DATA_FILE,
    SWEIGHT_FILE,
]

required_mc_files = [
    "dps_ccbar_9to30_usual.root",
    "dps_ccbar_30to50_usual.root",
    "dps_ccbar_50to100_usual.root",
    "dps_bbbar_9to30_usual.root",
    "dps_bbbar_30to50_usual.root",
    "dps_bbbar_50to100_usual.root",
    "sps_3fs_ccbar_25to100_usual.root",
    "sps_3fs_bbbar_25to100_usual.root",
]


# ----------------------------------------------------------------
# Efficiency inputs
# ----------------------------------------------------------------

required_efficiency_files = [
    f"efficiencies_{component}_differential_final_jpsi_{year}.root"
    for year in EFFICIENCY_YEARS
    for component in (
        "DPS-ccbar",
        "DPS-bbbar",
        "SPS-ccbar",
        "SPS-bbbar",
    )
]


# Prefer the EOS copy because this notebook is normally executed
# in SWAN/Jupyter and EOS visibility is stable there.
EFFICIENCY_DIR_CANDIDATES = [
    EFFICIENCY_DIR,
    DATA_DIR / "Data" / "efficiency_final_2017_2018",

    Path(
        "/eos/home-j/jgomespi/SWAN_projects/"
        "Associated_Production_JPsi_Dstar/"
        "Template_fit/Data/"
        "efficiency_final_2017_2018"
    ),

    Path(
        "/afs/cern.ch/work/j/jgomespi/private/workspace/"
        "JPsiDstar/Efficiency_Quarkonia_OpenCharm/"
        "output/efficiency/final"
    ),
]


def directory_contains_all_files(directory, filenames):
    directory = Path(directory)

    if not directory.is_dir():
        return False

    return all(
        (directory / filename).is_file()
        for filename in filenames
    )


EFFICIENCY_DIR = None

for candidate in EFFICIENCY_DIR_CANDIDATES:

    if directory_contains_all_files(
        candidate,
        required_efficiency_files,
    ):
        EFFICIENCY_DIR = candidate
        break


if EFFICIENCY_DIR is None:

    details = []

    for candidate in EFFICIENCY_DIR_CANDIDATES:

        missing_here = [
            filename
            for filename in required_efficiency_files
            if not (candidate / filename).is_file()
        ]

        details.append(
            "\nDirectory:\n"
            f"  {candidate}\n"
            "Missing:\n  "
            + "\n  ".join(missing_here)
        )

    raise FileNotFoundError(
        "Could not find a complete validated 2017+2018 "
        "efficiency set in any allowed location.\n"
        + "\n".join(details)
    )


# ----------------------------------------------------------------
# Validate data and selected MC
# ----------------------------------------------------------------

missing_main = [
    str(data_root_dir / filename)
    for filename in (
        required_data_files
        + required_mc_files
    )
    if not (
        data_root_dir / filename
    ).is_file()
]


if missing_main:

    raise FileNotFoundError(
        "Missing mandatory data/MC inputs:\n  "
        + "\n  ".join(missing_main)
    )


# ----------------------------------------------------------------
# Validate final efficiency ROOTs
# ----------------------------------------------------------------

missing_eff = [
    str(EFFICIENCY_DIR / filename)
    for filename in required_efficiency_files
    if not (
        EFFICIENCY_DIR / filename
    ).is_file()
]


if missing_eff:

    raise FileNotFoundError(
        "Missing mandatory 2017+2018 efficiency inputs:\n  "
        + "\n  ".join(missing_eff)
    )


# ----------------------------------------------------------------
# Validate full Run-2 mixing inputs
# ----------------------------------------------------------------

mixing_files = sorted(
    MIXING_INPUT_DIR.glob(
        MIXING_INPUT_GLOB
    )
)


if not mixing_files:

    raise FileNotFoundError(
        "No mixing inputs found under:\n  "
        f"{MIXING_INPUT_DIR / MIXING_INPUT_GLOB}"
    )


# ----------------------------------------------------------------
# Hard sanity checks
# ----------------------------------------------------------------

if len(required_data_files) != 2:
    raise RuntimeError(
        "Internal preflight error: expected exactly "
        "data ROOT + sWeight ROOT"
    )

if len(required_mc_files) != 8:
    raise RuntimeError(
        "Internal preflight error: expected exactly "
        "8 selected-MC ROOT files"
    )

if len(required_efficiency_files) != 8:
    raise RuntimeError(
        "Internal preflight error: expected exactly "
        "8 final efficiency ROOT files"
    )

if set(EFFICIENCY_YEARS) != {
    "2017",
    "2018",
}:
    raise RuntimeError(
        "This notebook expects exactly the validated "
        "2017+2018 efficiency response. "
        f"Found EFFICIENCY_YEARS={EFFICIENCY_YEARS}"
    )


# ----------------------------------------------------------------
# Report
# ----------------------------------------------------------------

print("=" * 72)
print("INPUT PREFLIGHT: PASS")
print("=" * 72)

print()
print("Data directory:")
print(" ", data_root_dir)

print()
print("Efficiency directory selected:")
print(" ", EFFICIENCY_DIR)

print()
print(
    "data/sWeight ROOTs:",
    len(required_data_files),
)

print(
    "selected-MC ROOTs:",
    len(required_mc_files),
)

print(
    "final efficiency ROOTs:",
    len(required_efficiency_files),
)

print(
    "mixing coffea files:",
    len(mixing_files),
)


# ----------------------------------------------------------------
# Informational checks
# ----------------------------------------------------------------

print()
print("Efficiency response years:")
print(" ", list(EFFICIENCY_YEARS))

print(
    "Efficiency-response luminosity [fb^-1]:",
    sum(
        LUMI_BY_YEAR_FB[year]
        for year in EFFICIENCY_YEARS
    ),
)

print(
    "Full Run-2 data luminosity [fb^-1]:",
    LUMI_RUN2_FB_INV,
)


if not SHAPE_SYSTEMATICS_DIR.is_dir():

    warnings.warn(
        "Differential shape-systematics directory is absent. "
        "This does NOT block the nominal/statistical fit. "
        "The final systematic-completeness audit must remain "
        "non-final until those inputs are supplied."
    )


print()
print("Preflight contract:")
print("  full Run-2 data                : READY")
print("  full Run-2 sWeights            : READY")
print("  selected SPS/DPS MC            : READY")
print("  2017 final efficiency response : READY")
print("  2018 final efficiency response : READY")
print("  full Run-2 mixed-event pool    : READY")

if SHAPE_SYSTEMATICS_DIR.is_dir():
    print(
        "  differential shape syst.      : READY"
    )
else:
    print(
        "  differential shape syst.      : NOT AVAILABLE "
        "(non-blocking for nominal fit)"
    )

print()
print("=" * 72)


## Full Run-2 data and sWeights

In [ ]:
BRANCH_CANDIDATES = {
    "jpsi_mass": ["jpsi_mass", "Jpsi_mass", "dimu_mass", "pair_data_jpsi_mass"],
    "dstar_deltam": ["dstar_deltam", "dstar_deltamr", "dstar_mass", "pair_data_dstar_deltamr"],
    "jpsi_dl": ["jpsi_dl", "Jpsi_dl", "dimu_dl", "pair_data_jpsi_dl"],
    "delta_phi": ["jpsi_dstar_deltaphi", "delta_phi", "deltaphi", "pair_data_delta_phi"],
    "delta_y": ["jpsi_dstar_deltarap", "jpsi_dstar_deltay", "delta_y", "deltarap", "pair_data_delta_y"],
    "jpsi_rap": ["jpsi_rap", "jpsi_y", "dimu_rap", "pair_data_jpsi_rap"],
    "jpsi_phi": ["jpsi_phi", "dimu_phi", "pair_data_jpsi_phi"],
    "jpsi_pt": ["jpsi_pt", "dimu_pt", "pair_data_jpsi_pt"],
    "dstar_rap": ["dstar_rap", "dstar_y", "pair_data_dstar_rap"],
    "dstar_phi": ["dstar_phi", "pair_data_dstar_phi"],
    "dstar_pt": ["dstar_pt", "pair_data_dstar_pt"],
    "sweight": ["sweight", "sWeight", "signal_weight", "sw", "weight"],
}


def canonical_name(name):
    return re.sub(r"[^a-z0-9]", "", str(name).lower())


def fold_delta_phi(values):
    values = np.asarray(values, dtype=float)
    return np.abs(np.arctan2(np.sin(values), np.cos(values)))


def find_file(filename, roots=None):
    roots = list(roots or []) + [
        DATA_DIR / "Data",
        DATA_DIR,
        Path.cwd(),
        Path.cwd()/"Template_fit"/"Data",
        Path("/mnt/data"),
    ]
    seen = set()
    for raw_root in roots:
        root = Path(raw_root).expanduser()
        try:
            root = root.resolve()
        except Exception:
            pass
        if root in seen or not root.exists():
            continue
        seen.add(root)
        direct = root / filename
        if direct.exists():
            return direct
        matches = list(root.rglob(filename))
        if matches:
            return matches[0]
    return None


def open_first_ttree(path, preferred=("asso", "splot", "Events", "tree")):
    root_file = uproot.open(path)
    classnames = root_file.classnames()
    for preferred_name in preferred:
        for key, classname in classnames.items():
            if key.split(";")[0] == preferred_name and "TTree" in classname:
                return root_file, root_file[key]
    for key, classname in classnames.items():
        if "TTree" in classname:
            return root_file, root_file[key]
    root_file.close()
    raise RuntimeError(f"No TTree found in {path}")


def resolve_branch(tree, logical, required=True):
    branches = list(tree.keys())
    canonical = {canonical_name(branch): branch for branch in branches}
    for candidate in BRANCH_CANDIDATES.get(logical, [logical]):
        if candidate in branches:
            return candidate
        key = canonical_name(candidate)
        if key in canonical:
            return canonical[key]
    if required:
        raise KeyError(f"Could not resolve {logical}; available={branches[:80]}")
    return None


def read_logical_dataframe(path, logical_names, fixed=None):
    root_file, tree = open_first_ttree(path)
    aliases = {logical: resolve_branch(tree, logical) for logical in logical_names}
    for logical, branch in (fixed or {}).items():
        if branch not in tree.keys():
            root_file.close()
            raise KeyError(f"Missing fixed branch {branch!r} in {path}")
        aliases[logical] = branch
    arrays = tree.arrays(sorted(set(aliases.values())), library="pd")
    root_file.close()
    inverse = {branch: logical for logical, branch in aliases.items()}
    return arrays.rename(columns=inverse).reset_index(drop=True), aliases


data_path = find_file(DATA_FILE)
sweight_path = find_file(SWEIGHT_FILE)
if data_path is None or sweight_path is None:
    raise FileNotFoundError("Could not locate the Run-2 data and sWeight ROOT files")

logical_data = [
    "jpsi_mass", "dstar_deltam", "jpsi_dl", "delta_phi", "delta_y",
    "jpsi_rap", "jpsi_phi", "jpsi_pt", "dstar_rap", "dstar_phi", "dstar_pt",
]
data_raw, _ = read_logical_dataframe(data_path, logical_data)
sweight_raw, _ = read_logical_dataframe(sweight_path, ["sweight"])

if len(data_raw) != REFERENCE_RAW_RUN2_ENTRIES:
    raise RuntimeError(
        f"Unexpected raw Run-2 entry count: {len(data_raw)} "
        f"(expected {REFERENCE_RAW_RUN2_ENTRIES})"
    )
if len(sweight_raw) != REFERENCE_SPLOT_ENTRIES:
    raise RuntimeError(
        f"Unexpected sWeight entry count: {len(sweight_raw)} "
        f"(expected {REFERENCE_SPLOT_ENTRIES})"
    )

data_raw["delta_phi"] = fold_delta_phi(data_raw["delta_phi"])
data_raw["delta_y"] = np.abs(data_raw["delta_y"].to_numpy(float))

splot_mask = (
    (data_raw["jpsi_mass"] > JPSI_MASS_MIN)
    & (data_raw["jpsi_mass"] < JPSI_MASS_MAX)
    & (data_raw["jpsi_dl"] > JPSI_DL_MIN)
    & (data_raw["jpsi_dl"] < JPSI_DL_MAX)
    & (data_raw["dstar_deltam"] > 0.13957)
    & (data_raw["dstar_deltam"] < 0.16000)
)
data_sw = data_raw.loc[splot_mask].reset_index(drop=True).copy()
sweights = sweight_raw["sweight"].to_numpy(float)
if len(data_sw) != len(sweights):
    raise ValueError(
        f"Selected data ({len(data_sw)}) and sWeights ({len(sweights)}) differ"
    )
if len(data_sw) != REFERENCE_SPLOT_ENTRIES:
    raise RuntimeError(
        f"Unexpected sPlot-selected data count: {len(data_sw)} "
        f"(expected {REFERENCE_SPLOT_ENTRIES})"
    )
data_sw["sweight"] = sweights

# Enforce the same fiducial kinematics used for MC without breaking the
# event-by-event sWeight alignment. The current input is already fully
# fiducial; fail loudly if a future input is not.
data_fiducial_mask = (
    (data_sw["jpsi_pt"] > JPSI_PT_MIN)
    & (data_sw["jpsi_pt"] < JPSI_PT_MAX)
    & (np.abs(data_sw["jpsi_rap"]) < JPSI_ABSY_MAX)
    & (data_sw["dstar_pt"] > DSTAR_PT_MIN)
    & (data_sw["dstar_pt"] < DSTAR_PT_MAX)
    & (np.abs(data_sw["dstar_rap"]) < DSTAR_ABSY_MAX)
)
if not bool(np.all(data_fiducial_mask)):
    raise RuntimeError(
        f"{np.count_nonzero(~data_fiducial_mask)} sPlot-matched data events "
        "fall outside the MC fiducial phase space; rebuild aligned sWeights."
    )

selection_summary = pd.DataFrame([
    ("Raw Run-2 entries", len(data_raw)),
    ("sWeight entries", len(sweights)),
    ("sPlot-matched entries", len(data_sw)),
    ("Data outside MC fiducial phase space", int(np.count_nonzero(~data_fiducial_mask))),
    ("Sum of sWeights", float(np.sum(sweights))),
    ("sqrt(sum(sWeight^2))", float(np.sqrt(np.sum(sweights**2)))),
    ("Negative-sWeight fraction", float(np.mean(sweights < 0.0))),
    ("sWeight effective entries", weighted_effective_entries(sweights)),
], columns=["quantity", "value"])
print(selection_summary.to_string(index=False))
selection_summary.to_csv(OUTDIR/"splot_matching_summary.csv", index=False)

# Necessary diagnostic for using the sPlot projection in the physics variables.
rows = []
for discriminant in ["jpsi_mass", "dstar_deltam", "jpsi_dl"]:
    for observable in ["delta_phi", "delta_y"]:
        rho, pvalue = spearmanr(
            data_sw[discriminant].to_numpy(float),
            data_sw[observable].to_numpy(float),
            nan_policy="omit",
        )
        rows.append({
            "discriminant": discriminant,
            "observable": observable,
            "spearman_rho": float(rho),
            "pvalue": float(pvalue),
        })
splot_correlation_table = pd.DataFrame(rows)
splot_correlation_table.to_csv(OUTDIR/"splot_observable_correlations.csv", index=False)

rho_matrix = (
    splot_correlation_table.pivot(
        index="discriminant", columns="observable", values="spearman_rho"
    )
    .reindex(index=["jpsi_mass", "dstar_deltam", "jpsi_dl"],
             columns=["delta_phi", "delta_y"])
)
fig, ax = plt.subplots(figsize=(6.8, 4.8), layout="constrained")
im = ax.imshow(rho_matrix.to_numpy(float), vmin=-0.15, vmax=0.15, aspect="auto", cmap="coolwarm")
ax.set_xticks(np.arange(2), [r"$|\Delta\phi|$", r"$|\Delta y|$"])
ax.set_yticks(np.arange(3), [r"$m(J/\psi)$", r"$\Delta m(D^*)$", r"$\ell_{J/\psi}$"])
for iy in range(3):
    for ix in range(2):
        ax.text(ix, iy, f"{rho_matrix.iloc[iy, ix]:+.3f}", ha="center", va="center", fontsize=11)
cbar = fig.colorbar(im, ax=ax, shrink=0.86)
cbar.set_label("Spearman $\\rho$")
ax.set_title("sPlot discriminant–physics-variable correlation audit", pad=12)
save_and_show(fig, "splot_correlation_audit", OUTDIR)
if np.nanmax(np.abs(splot_correlation_table["spearman_rho"])) >= 0.10:
    warnings.warn(
        "At least one signal-fit discriminant has |Spearman rho| >= 0.10 with a physics observable. "
        "The sPlot factorization assumption needs an explicit closure validation before this fit is reportable."
    )


## Selected SPS/DPS Monte Carlo

In [ ]:
SAMPLE_INFO = {
    "DPS-ccbar-25to30": {
        "component": "DPS-ccbar", "kind": "DPS", "flavour": "ccbar",
        "pt_low": 25.0, "pt_high": 30.0,
        "root_file": "dps_ccbar_9to30_usual.root", "csv_file": "data_dps_ccbar_25to30.csv",
        "nevt": 4895653*(4.9/4.12), "xsec_barn": 373.3e-12,
    },
    "DPS-ccbar-30to50": {
        "component": "DPS-ccbar", "kind": "DPS", "flavour": "ccbar",
        "pt_low": 30.0, "pt_high": 50.0,
        "root_file": "dps_ccbar_30to50_usual.root", "csv_file": "data_dps_ccbar_30to50.csv",
        "nevt": 1095994*(1.0/1.3), "xsec_barn": 0.1351e-12,
    },
    "DPS-ccbar-50to100": {
        "component": "DPS-ccbar", "kind": "DPS", "flavour": "ccbar",
        "pt_low": 50.0, "pt_high": 100.0,
        "root_file": "dps_ccbar_50to100_usual.root", "csv_file": "data_dps_ccbar_50to100.csv",
        "nevt": 477085*(4.77/5.5), "xsec_barn": 0.001466e-12,
    },
    "DPS-bbbar-25to30": {
        "component": "DPS-bbbar", "kind": "DPS", "flavour": "bbbar",
        "pt_low": 25.0, "pt_high": 30.0,
        "root_file": "dps_bbbar_9to30_usual.root", "csv_file": "data_dps_bbbar_25to30.csv",
        "nevt": 502277*(502.28/752.0), "xsec_barn": 0.625e-12,
    },
    "DPS-bbbar-30to50": {
        "component": "DPS-bbbar", "kind": "DPS", "flavour": "bbbar",
        "pt_low": 30.0, "pt_high": 50.0,
        "root_file": "dps_bbbar_30to50_usual.root", "csv_file": "data_dps_bbbar_30to50.csv",
        "nevt": 239513*(239.51/233.0), "xsec_barn": 0.000702e-12,
    },
    "DPS-bbbar-50to100": {
        "component": "DPS-bbbar", "kind": "DPS", "flavour": "bbbar",
        "pt_low": 50.0, "pt_high": 100.0,
        "root_file": "dps_bbbar_50to100_usual.root", "csv_file": "data_dps_bbbar_50to100.csv",
        "nevt": 105746*(105.75/130.0), "xsec_barn": 0.00000771e-12,
    },
    "SPS-ccbar-25to100": {
        "component": "SPS-ccbar", "kind": "SPS", "flavour": "ccbar",
        "pt_low": 25.0, "pt_high": 100.0,
        "root_file": "sps_3fs_ccbar_25to100_usual.root", "csv_file": "data_sps_3fs_ccbar.csv",
        "nevt": 554631.676300578, "xsec_barn": 0.008139e-12,
    },
    "SPS-bbbar-25to100": {
        "component": "SPS-bbbar", "kind": "SPS", "flavour": "bbbar",
        "pt_low": 25.0, "pt_high": 100.0,
        "root_file": "sps_3fs_bbbar_25to100_usual.root", "csv_file": "data_sps_3fs_bbbar.csv",
        "nevt": 99168.64, "xsec_barn": 0.00570e-12,
    },
}

FIXED_DPS_PT_STITCH = {
    "DPS-ccbar-25to30": 1.0,
    "DPS-ccbar-30to50": 17.207198481273394,
    "DPS-ccbar-50to100": 276.6850750445472,
    "DPS-bbbar-25to30": 1.0,
    "DPS-bbbar-30to50": 12.015539297981329,
    "DPS-bbbar-50to100": 143.82641208838336,
    "SPS-ccbar-25to100": 1.0,
    "SPS-bbbar-25to100": 1.0,
}
STITCH_MODE = "derive_from_jpsi_pt"
STITCH_EDGE_WIDTH_GEV = 1.0


def standardize_csv_columns(frame):
    rename = {}
    for logical, candidates in BRANCH_CANDIDATES.items():
        for candidate in candidates:
            if candidate in frame.columns:
                rename[candidate] = logical
                break
    out = frame.rename(columns=rename).copy()
    if "tree_weight" not in out.columns:
        out["tree_weight"] = 1.0
    return out


def mc_fiducial_mask(frame):
    return (
        (frame["jpsi_pt"] > JPSI_PT_MIN) & (frame["jpsi_pt"] < JPSI_PT_MAX)
        & (np.abs(frame["jpsi_rap"]) < JPSI_ABSY_MAX)
        & (frame["dstar_pt"] > DSTAR_PT_MIN) & (frame["dstar_pt"] < DSTAR_PT_MAX)
        & (np.abs(frame["dstar_rap"]) < DSTAR_ABSY_MAX)
    )


def load_one_mc_sample(sample_name, info):
    root_path = find_file(info["root_file"])
    csv_path = find_file(info["csv_file"])
    logical = ["delta_phi", "delta_y", "jpsi_pt", "jpsi_rap", "dstar_pt", "dstar_rap"]
    if root_path is not None:
        frame, _ = read_logical_dataframe(
            root_path, logical, fixed={"tree_weight": MC_TREE_WEIGHT_BRANCH}
        )
        input_path = root_path
    elif csv_path is not None:
        frame = standardize_csv_columns(pd.read_csv(csv_path))
        input_path = csv_path
    else:
        raise FileNotFoundError(f"Missing ROOT/CSV input for {sample_name}")

    missing = [name for name in logical + ["tree_weight"] if name not in frame]
    if missing:
        raise KeyError(f"{sample_name} missing {missing} in {input_path}")

    frame["delta_phi"] = fold_delta_phi(frame["delta_phi"])
    frame["delta_y"] = np.abs(frame["delta_y"].to_numpy(float))
    before = len(frame)
    frame = frame.loc[mc_fiducial_mask(frame)].reset_index(drop=True)
    frame["sample"] = sample_name
    frame["component"] = info["component"]
    frame["kind"] = info["kind"]
    frame["flavour"] = info["flavour"]
    frame["pt_low"] = info["pt_low"]
    frame["pt_high"] = info["pt_high"]
    base = info["xsec_barn"]*(LUMI_RUN2_FB_INV/1.0e-15)/info["nevt"]
    frame["mc_weight_xsec_only"] = frame["tree_weight"].to_numpy(float)*base
    frame["pt_stitch_scale"] = 1.0
    frame["mc_weight"] = frame["mc_weight_xsec_only"]
    print(f"[MC] {sample_name:22s} fiducial={len(frame):7d}/{before:7d} input={input_path}")
    return frame


def apply_fixed_stitching(mc):
    out = mc.copy()
    out["pt_stitch_scale"] = out["sample"].map(FIXED_DPS_PT_STITCH).fillna(1.0)
    out["mc_weight"] = out["mc_weight_xsec_only"]*out["pt_stitch_scale"]
    table = pd.DataFrame([
        {"component": SAMPLE_INFO[name]["component"], "sample": name,
         "boundary": np.nan, "pt_stitch_scale": scale,
         "pt_stitch_rel_unc": np.nan, "method": "fixed"}
        for name, scale in FIXED_DPS_PT_STITCH.items()
    ])
    return out, table


def derive_stitching(mc, edge_width=1.0):
    out = mc.copy()
    records = []
    for component in ["DPS-ccbar", "DPS-bbbar"]:
        ordered = sorted(
            [name for name, info in SAMPLE_INFO.items() if info["component"] == component],
            key=lambda name: SAMPLE_INFO[name]["pt_low"],
        )
        scales = {ordered[0]: 1.0}
        rel_unc = {ordered[0]: 0.0}
        records.append({"component": component, "sample": ordered[0], "boundary": np.nan,
                        "pt_stitch_scale": 1.0, "pt_stitch_rel_unc": 0.0, "method": "anchor"})
        for left_name, right_name in zip(ordered[:-1], ordered[1:]):
            boundary = SAMPLE_INFO[right_name]["pt_low"]
            left = out[out["sample"] == left_name]
            right = out[out["sample"] == right_name]
            lm = (left["jpsi_pt"] >= boundary-edge_width) & (left["jpsi_pt"] < boundary)
            rm = (right["jpsi_pt"] >= boundary) & (right["jpsi_pt"] < boundary+edge_width)
            lw = left.loc[lm, "mc_weight_xsec_only"].to_numpy(float)
            rw = right.loc[rm, "mc_weight_xsec_only"].to_numpy(float)
            lbase = float(np.sum(lw))
            rsum = float(np.sum(rw))
            lsum = lbase*scales[left_name]
            if rsum > 0.0 and np.isfinite(lsum) and lbase != 0.0:
                scale = lsum/rsum
                method = "boundary_density_match"
                lstat = math.sqrt(float(np.sum(lw**2)))/abs(lbase)
                rstat = math.sqrt(float(np.sum(rw**2)))/abs(rsum)
                uncertainty = math.sqrt(rel_unc[left_name]**2 + lstat**2 + rstat**2)
            else:
                scale = FIXED_DPS_PT_STITCH[right_name]
                method = "fallback_fixed"
                uncertainty = np.nan
            scales[right_name] = scale
            rel_unc[right_name] = uncertainty
            records.append({"component": component, "sample": right_name, "boundary": boundary,
                            "pt_stitch_scale": scale, "pt_stitch_rel_unc": uncertainty,
                            "method": method})
        for sample_name, scale in scales.items():
            out.loc[out["sample"] == sample_name, "pt_stitch_scale"] = scale
    out.loc[out["kind"] == "SPS", "pt_stitch_scale"] = 1.0
    out["mc_weight"] = out["mc_weight_xsec_only"]*out["pt_stitch_scale"]
    return out, pd.DataFrame(records)


mc_raw = pd.concat(
    [load_one_mc_sample(name, info) for name, info in SAMPLE_INFO.items()],
    ignore_index=True, sort=False,
)
if STITCH_MODE == "derive_from_jpsi_pt":
    mc_df, stitch_table = derive_stitching(mc_raw, STITCH_EDGE_WIDTH_GEV)
else:
    mc_df, stitch_table = apply_fixed_stitching(mc_raw)

stitch_audit = (
    mc_df.groupby(["component", "sample"], sort=False)["mc_weight"]
    .agg(sumw="sum", sumw2=lambda values: float(np.sum(np.asarray(values, float)**2)))
    .reset_index()
)
stitch_audit["effective_entries"] = safe_divide(
    stitch_audit["sumw"]**2, stitch_audit["sumw2"], default=0.0
)
stitch_audit["component_fraction"] = stitch_audit["sumw"] / stitch_audit.groupby(
    "component"
)["sumw"].transform("sum")

print("\nNominal DPS stitching:")
for _, row in stitch_table.iterrows():
    if str(row.get("component","")).startswith("DPS"):
        unc = row.get("pt_stitch_rel_unc", np.nan)
        suffix = f" ± {100*unc:.1f}% (stat)" if np.isfinite(unc) else ""
        print(f"  {row['sample']:22s}: {row['pt_stitch_scale']:.6g}{suffix}")
stitch_table.to_csv(OUTDIR/"dps_pt_stitching_used.csv", index=False)
stitch_audit.to_csv(OUTDIR/"dps_pt_stitching_audit.csv", index=False)


## Validated 2017+2018 efficiency maps

In [ ]:
class Hist2DMap:
    def __init__(self, values, xedges, yedges, name):
        self.values = np.asarray(values, dtype=float)
        self.xedges = np.asarray(xedges, dtype=float)
        self.yedges = np.asarray(yedges, dtype=float)
        self.name = str(name)
        if self.values.shape != (len(self.xedges)-1, len(self.yedges)-1):
            raise ValueError(f"Bad map shape for {name}: {self.values.shape}")

    @classmethod
    def from_uproot(cls, obj, name):
        values, xedges, yedges = obj.to_numpy(flow=False)
        return cls(values, xedges, yedges, name)

    def compatible(self, other):
        return np.allclose(self.xedges, other.xedges) and np.allclose(self.yedges, other.yedges)

    def bin_indices(self, x, y):
        x = np.asarray(x, dtype=float)
        y = np.asarray(y, dtype=float)
        ix = np.searchsorted(self.xedges, x, side="right") - 1
        iy = np.searchsorted(self.yedges, y, side="right") - 1
        ix = np.where(x == self.xedges[-1], self.values.shape[0]-1, ix)
        iy = np.where(y == self.yedges[-1], self.values.shape[1]-1, iy)
        valid = (
            np.isfinite(x) & np.isfinite(y)
            & (ix >= 0) & (ix < self.values.shape[0])
            & (iy >= 0) & (iy < self.values.shape[1])
        )
        if not np.all(valid):
            raise ValueError(
                f"[eff] {self.name}: {np.count_nonzero(~valid)}/{valid.size} entries outside map coverage"
            )
        return ix.astype(np.int32), iy.astype(np.int32)

    def eval(self, x, y, values_override=None):
        ix, iy = self.bin_indices(x, y)
        values = self.values if values_override is None else np.asarray(values_override, dtype=float)
        return values[ix, iy]


EFFICIENCY_SEARCH_DIRS = [
    EFFICIENCY_DIR,
    EFFICIENCY_DIR/"final",
    DATA_DIR/"Data"/"efficiency",
    DATA_DIR/"Data"/"efficiency"/"final",
    Path.cwd()/"efficiency",
    Path.cwd()/"output"/"efficiency",
]
EFFICIENCY_SOURCE_CACHE = {}
EFFICIENCY_FILE_AUDIT = []
EFFICIENCY_BIN_AUDIT = []


def efficiency_filename(source, year):
    return f"efficiencies_{source}_differential_final_jpsi_{year}.root"


def find_efficiency_file(source, year):
    filename = efficiency_filename(source, year)
    for root in EFFICIENCY_SEARCH_DIRS:
        root = Path(root).expanduser()
        for candidate in (root/filename, root/"final"/filename):
            if candidate.is_file():
                return candidate.resolve()
    for root in EFFICIENCY_SEARCH_DIRS:
        root = Path(root).expanduser()
        if root.is_dir():
            matches = sorted(root.rglob(filename))
            if matches:
                return matches[0].resolve()
    return None



def validate_efficiency_campaign():
    """Require the complete validated 2017+2018 response for all four components."""
    sources = sorted(set(EFFICIENCY_SOURCE_FOR_COMPONENT.values()))
    missing = [
        f"{source}:{year}"
        for source in sources
        for year in EFFICIENCY_YEARS
        if find_efficiency_file(source, year) is None
    ]
    if missing:
        raise FileNotFoundError(
            "Mandatory 2017+2018 efficiency maps are missing: "
            + ", ".join(missing)
        )

    print("[eff] data period: full Run 2")
    print("[eff] response years:", list(EFFICIENCY_YEARS))
    print(
        "[eff] response luminosity:",
        f"{LUMI_EFFICIENCY_RESPONSE_FB_INV:.2f} fb^-1",
    )
    print("[eff] 2016 response pending: yes")
    return True


EFFICIENCY_CAMPAIGN_COMPLETE_1718 = validate_efficiency_campaign()


def _read_hist_if_present(root_file, key):
    if key not in root_file:
        return None
    obj = root_file[key]
    out = obj.to_numpy(flow=False)
    if len(out) != 3:
        return None
    return Hist2DMap.from_uproot(obj, key)


def load_efficiency_file(path):
    result = {}
    with uproot.open(path) as root_file:
        keys = {key.split(";")[0] for key in root_file.keys(recursive=True)}
        for factor in EFFICIENCY_REQUIRED:
            central = _read_hist_if_present(root_file, factor)
            if central is None:
                continue
            info = {"central": central}
            for label, key in [
                ("err_up", f"{factor}_err_up"),
                ("err_down", f"{factor}_err_down"),
                ("n_eff", f"{factor}_n_eff"),
            ]:
                h = _read_hist_if_present(root_file, key)
                if h is not None:
                    info[label] = h
            raw = {}
            for suffix in ("num_sumw", "num_sumw2", "den_sumw", "den_sumw2"):
                key = f"raw/{factor}_{suffix}"
                h = _read_hist_if_present(root_file, key)
                if h is not None:
                    raw[suffix] = h
            info["raw"] = raw
            result[factor] = info
    return result


def _weighted_ratio_variance_from_raw(info):
    raw = info.get("raw", {})
    needed = {"num_sumw", "num_sumw2", "den_sumw", "den_sumw2"}
    if not needed.issubset(raw):
        return None
    e = info["central"].values
    D = raw["den_sumw"].values
    Qn = raw["num_sumw2"].values
    Qd = raw["den_sumw2"].values
    # Delta-method variance for a weighted subset ratio N/D.
    # numerator events are a subset of denominator events.
    qfail = np.maximum(Qd-Qn, 0.0)
    var = safe_divide((1.0-e)**2*Qn + e**2*qfail, D**2, default=np.nan)
    return var


def load_efficiency_source(source):
    if source in EFFICIENCY_SOURCE_CACHE:
        return EFFICIENCY_SOURCE_CACHE[source]
    years = {}
    for year in EFFICIENCY_YEARS:
        path = find_efficiency_file(source, year)
        if path is None:
            raise FileNotFoundError(
                f"Mandatory efficiency ROOT missing: {source}, {year}"
            )
        maps = load_efficiency_file(path)
        missing = [name for name in EFFICIENCY_REQUIRED if name not in maps]
        if missing:
            raise KeyError(
                f"{source}/{year}: missing mandatory efficiency maps {missing}"
            )
        EFFICIENCY_FILE_AUDIT.append({
            "source": source, "year": year, "file_found": True,
            "path": str(path), "missing": "",
        })
        years[year] = {"path": path, "maps": maps}
        print(f"[eff] loaded {source}, {year}: {path}")

        for factor, info in maps.items():
            raw_var = _weighted_ratio_variance_from_raw(info)
            enc = None
            if "err_up" in info and "err_down" in info:
                enc = 0.5*(info["err_up"].values + info["err_down"].values)
            for index in np.ndindex(info["central"].values.shape):
                row = {
                    "source": source,
                    "year": year,
                    "factor": factor,
                    "bin": str(index),
                    "eff": float(info["central"].values[index]),
                    "n_eff": float(info["n_eff"].values[index]) if "n_eff" in info else np.nan,
                    "encoded_sigma_sym": float(enc[index]) if enc is not None else np.nan,
                    "raw_delta_sigma": float(np.sqrt(max(raw_var[index], 0.0))) if raw_var is not None and np.isfinite(raw_var[index]) else np.nan,
                }
                EFFICIENCY_BIN_AUDIT.append(row)

    if set(years) != set(EFFICIENCY_YEARS):
        raise RuntimeError(
            f"{source}: loaded years {sorted(years)} do not match "
            f"required years {list(EFFICIENCY_YEARS)}"
        )
    EFFICIENCY_SOURCE_CACHE[source] = years
    return years


def factor_coordinates(frame, factor):
    jpt = frame["jpsi_pt"].to_numpy(float)
    jy = np.abs(frame["jpsi_rap"].to_numpy(float))
    dpt = frame["dstar_pt"].to_numpy(float)
    dy = np.abs(frame["dstar_rap"].to_numpy(float))
    if factor in {"acc_dimu", "eff_cuts_dimu", "eff_trigger"}:
        return jpt, jy
    if factor in {"acc_dstar", "eff_cuts_dstar"}:
        return dpt, dy
    if factor == "eff_asso_pt":
        return jpt, dpt
    if factor == "eff_asso_rap":
        return jy, dy
    raise KeyError(factor)


def sample_factor_map(info, z, factor):
    central = info["central"].values
    up = info.get("err_up")
    down = info.get("err_down")
    if up is None or down is None:
        return central.copy()

    z = np.asarray(z, dtype=float)
    if z.shape != central.shape:
        raise ValueError("Efficiency-replica z shape mismatch")

    varied = central + np.where(z >= 0.0, z*up.values, z*down.values)

    # acc_dimu and acc_dstar are weighted response ratios, not strict
    # binomial probabilities. Values slightly above one are legitimate.
    if factor in {"acc_dimu", "acc_dstar"}:
        return np.maximum(varied, EFFICIENCY_FLOOR)

    return np.clip(varied, EFFICIENCY_FLOOR, 1.0)


def draw_efficiency_z(rng):
    replica = {}
    for source in sorted(set(EFFICIENCY_SOURCE_FOR_COMPONENT.values())):
        years = load_efficiency_source(source)
        for year, bundle in years.items():
            for factor in EFFICIENCY_FACTORS_USED:
                if factor not in bundle["maps"]:
                    continue
                shape = bundle["maps"][factor]["central"].values.shape
                replica[(source, year, factor)] = rng.normal(size=shape)
    return replica


def evaluate_efficiency_product_year(
    frame,
    source,
    year,
    z_replica=None,
    association_mode=None,
):
    """Evaluate the six-factor efficiency product for one response year."""
    year = str(year)
    if year not in EFFICIENCY_YEARS:
        raise ValueError(
            f"Efficiency year {year} is outside {list(EFFICIENCY_YEARS)}"
        )

    years = load_efficiency_source(source)
    bundle = years[year]
    maps = bundle["maps"]

    association_mode = (
        ASSOCIATION_EFF_MODE if association_mode is None else association_mode
    )
    factors = [
        "acc_dimu",
        "acc_dstar",
        "eff_cuts_dimu",
        "eff_cuts_dstar",
        "eff_trigger",
        "eff_asso_pt" if association_mode == "pt" else "eff_asso_rap",
    ]

    total_eff = np.ones(len(frame), dtype=float)
    for factor in factors:
        if factor not in maps:
            raise KeyError(f"Missing mandatory map {source}/{year}/{factor}")
        info = maps[factor]
        x, y = factor_coordinates(frame, factor)
        values = info["central"].values
        key = (source, year, factor)
        if z_replica is not None and key in z_replica:
            values = sample_factor_map(info, z_replica[key], factor)
        total_eff *= info["central"].eval(x, y, values_override=values)

    if np.any(~np.isfinite(total_eff)) or np.any(total_eff <= 0.0):
        raise ValueError(f"Invalid efficiency product for {source}/{year}")
    return np.maximum(total_eff, EFFICIENCY_FLOOR)


def evaluate_efficiency_product(
    frame,
    source,
    z_replica=None,
    association_mode=None,
):
    """Luminosity-weighted 2017+2018 detector response for full Run-2 inputs.

    The product is formed separately within each year and the two yearly
    products are then averaged.  This avoids multiplying luminosity-averaged
    factors, which is not equivalent.
    """
    weighted = np.zeros(len(frame), dtype=float)
    total_lumi = 0.0

    for year in EFFICIENCY_YEARS:
        lumi = float(EFFICIENCY_YEAR_WEIGHTS[year])
        weighted += lumi * evaluate_efficiency_product_year(
            frame,
            source,
            year,
            z_replica=z_replica,
            association_mode=association_mode,
        )
        total_lumi += lumi

    if total_lumi <= 0.0:
        raise RuntimeError("Non-positive efficiency-response luminosity")

    combined = weighted / total_lumi
    if np.any(~np.isfinite(combined)) or np.any(combined <= 0.0):
        raise ValueError(f"Invalid combined efficiency product for {source}")
    return np.maximum(combined, EFFICIENCY_FLOOR)


def validate_loaded_efficiency_contract():
    """Hard gate on axes, finite values and N_eff for the nominal maps."""
    rows = []

    for source in sorted(set(EFFICIENCY_SOURCE_FOR_COMPONENT.values())):
        years = load_efficiency_source(source)
        ref_maps = years[EFFICIENCY_YEARS[0]]["maps"]

        for year in EFFICIENCY_YEARS:
            maps = years[year]["maps"]

            for factor in EFFICIENCY_REQUIRED:
                info = maps[factor]
                central = info["central"]

                if not np.all(np.isfinite(central.values)):
                    raise ValueError(f"{source}/{year}/{factor}: non-finite central values")
                if np.any(central.values <= 0.0):
                    raise ValueError(f"{source}/{year}/{factor}: non-positive central values")

                if factor not in {"acc_dimu", "acc_dstar"} and np.any(central.values > 1.0 + 1e-10):
                    raise ValueError(
                        f"{source}/{year}/{factor}: strict efficiency above one"
                    )

                for label in ("err_up", "err_down", "n_eff"):
                    if label not in info:
                        raise KeyError(f"{source}/{year}/{factor}: missing {label}")
                    if not central.compatible(info[label]):
                        raise ValueError(
                            f"{source}/{year}/{factor}: {label} axes differ from central"
                        )

                if year != EFFICIENCY_YEARS[0]:
                    ref = ref_maps[factor]["central"]
                    if not ref.compatible(central):
                        raise ValueError(
                            f"{source}/{factor}: 2017/2018 axes are incompatible"
                        )

                min_neff = float(np.nanmin(info["n_eff"].values))
                rows.append({
                    "source": source,
                    "year": year,
                    "factor": factor,
                    "min_neff": min_neff,
                })

                if factor in EFFICIENCY_FACTORS_USED and min_neff < MIN_EFFICIENCY_MAP_NEFF:
                    raise RuntimeError(
                        f"{source}/{year}/{factor}: min N_eff={min_neff:.3f} "
                        f"< {MIN_EFFICIENCY_MAP_NEFF}"
                    )

    audit = pd.DataFrame(rows)
    audit.to_csv(OUTDIR/"efficiency_final_contract_audit.csv", index=False)
    return audit


# Force loading now so the audit is produced before the fit.
for _source in sorted(set(EFFICIENCY_SOURCE_FOR_COMPONENT.values())):
    load_efficiency_source(_source)

EFFICIENCY_FILE_AUDIT_DF = pd.DataFrame(EFFICIENCY_FILE_AUDIT)
EFFICIENCY_BIN_AUDIT_DF = pd.DataFrame(EFFICIENCY_BIN_AUDIT)
found = int(EFFICIENCY_FILE_AUDIT_DF.get("file_found", pd.Series(dtype=bool)).fillna(False).sum()) if len(EFFICIENCY_FILE_AUDIT_DF) else 0
print(f"[eff] loaded {found} efficiency ROOT file entries")
if len(EFFICIENCY_BIN_AUDIT_DF):
    finite_neff = EFFICIENCY_BIN_AUDIT_DF["n_eff"].to_numpy(float)
    finite_neff = finite_neff[np.isfinite(finite_neff)]
    if len(finite_neff):
        print(f"[eff] encoded map-bin N_eff range: {np.min(finite_neff):.2f} -- {np.max(finite_neff):.2f}")
EFFICIENCY_FILE_AUDIT_DF.to_csv(OUTDIR/"efficiency_file_audit.csv", index=False)
EFFICIENCY_BIN_AUDIT_DF.to_csv(OUTDIR/"efficiency_bin_statistical_audit.csv", index=False)

if found != 8:
    raise RuntimeError(f"Expected exactly 8 final efficiency ROOT entries, found {found}")

EFFICIENCY_CONTRACT_AUDIT = validate_loaded_efficiency_contract()
print(
    "[eff] final 2017+2018 contract: PASS; "
    f"minimum nominal-map N_eff={EFFICIENCY_CONTRACT_AUDIT.loc[EFFICIENCY_CONTRACT_AUDIT['factor'].isin(EFFICIENCY_FACTORS_USED), 'min_neff'].min():.2f}"
)


## Nominal response problem and statistical covariance

In [ ]:
COMPONENTS = ["DPS-ccbar", "DPS-bbbar", "SPS-ccbar", "SPS-bbbar"]
COMPONENT_INDEX = {name: i for i, name in enumerate(COMPONENTS)}


def make_spec(name):
    edges = BINNING[name]
    return {
        "name": name,
        "dphi_edges": np.asarray(edges["dphi"], dtype=float),
        "dy_edges": np.asarray(edges["dy"], dtype=float),
    }


def assign_bins(frame, spec):
    x = frame["delta_phi"].to_numpy(float)
    y = frame["delta_y"].to_numpy(float)
    xe = spec["dphi_edges"]
    ye = spec["dy_edges"]
    ix = np.searchsorted(xe, x, side="right") - 1
    iy = np.searchsorted(ye, y, side="right") - 1
    ix = np.where(x == xe[-1], len(xe)-2, ix)
    iy = np.where(y == ye[-1], len(ye)-2, iy)
    valid = (
        np.isfinite(x) & np.isfinite(y)
        & (ix >= 0) & (ix < len(xe)-1)
        & (iy >= 0) & (iy < len(ye)-1)
    )
    ny = len(ye)-1
    flat = ix*ny + iy
    return flat.astype(np.int32), valid, (len(xe)-1)*ny


def helmert_contrast(n):
    if n < 2:
        raise ValueError("Need at least two bins")
    H = np.zeros((n-1, n), dtype=float)
    for i in range(1, n):
        H[i-1, :i] = 1.0/math.sqrt(i*(i+1.0))
        H[i-1, i] = -i/math.sqrt(i*(i+1.0))
    return H


def data_selected_contributions(frame):
    return frame["sweight"].to_numpy(float)/(LUMI_RUN2_PB_INV*BRANCHING_PRODUCT)


def central_stitch_scales():
    table = (
        mc_df[["sample", "pt_stitch_scale"]]
        .drop_duplicates("sample")
        .set_index("sample")["pt_stitch_scale"]
    )
    return {str(k): float(v) for k, v in table.items()}


CENTRAL_STITCH_SCALES = central_stitch_scales()
MC_SAMPLE_CACHE = {
    sample: mc_raw.loc[mc_raw["sample"] == sample].reset_index(drop=True)
    for sample in SAMPLE_INFO
}


def derive_replica_stitch_scales(multipliers):
    if STITCH_MODE != "derive_from_jpsi_pt":
        return CENTRAL_STITCH_SCALES.copy()
    scales = {}
    for component in ["DPS-ccbar", "DPS-bbbar"]:
        ordered = sorted(
            [name for name, info in SAMPLE_INFO.items() if info["component"] == component],
            key=lambda name: SAMPLE_INFO[name]["pt_low"],
        )
        scales[ordered[0]] = 1.0
        for left_name, right_name in zip(ordered[:-1], ordered[1:]):
            boundary = SAMPLE_INFO[right_name]["pt_low"]
            left = MC_SAMPLE_CACHE[left_name]
            right = MC_SAMPLE_CACHE[right_name]
            lm = (left["jpsi_pt"].to_numpy(float) >= boundary-STITCH_EDGE_WIDTH_GEV) & (left["jpsi_pt"].to_numpy(float) < boundary)
            rm = (right["jpsi_pt"].to_numpy(float) >= boundary) & (right["jpsi_pt"].to_numpy(float) < boundary+STITCH_EDGE_WIDTH_GEV)
            lw = left["mc_weight_xsec_only"].to_numpy(float) * multipliers[left_name]
            rw = right["mc_weight_xsec_only"].to_numpy(float) * multipliers[right_name]
            lsum = float(np.sum(lw[lm])) * scales[left_name]
            rsum = float(np.sum(rw[rm]))
            if rsum > 0.0 and np.isfinite(lsum):
                scales[right_name] = lsum/rsum
            else:
                scales[right_name] = CENTRAL_STITCH_SCALES[right_name]
    for sample, info in SAMPLE_INFO.items():
        if info["kind"] == "SPS":
            scales[sample] = 1.0
    return scales


def _component_from_histograms(selected_hist, fid_hist, fid_total):
    if fid_total <= 0.0 or not np.isfinite(fid_total):
        raise ValueError("Non-positive fiducial total")
    response = np.asarray(selected_hist, float)/fid_total
    fid_shape = np.asarray(fid_hist, float)/fid_total
    return response, fid_shape, float(fid_total)


def _build_component_states(problem, multipliers=None, stitch_scales=None, z_replica=None,
                            bootstrap_hist=True, region=None):
    spec = problem["spec"]
    region = CENTRAL_DPS_REGION if region is None else region
    n_bins = problem["n_bins"]
    selected_h = {c: np.zeros(n_bins) for c in COMPONENTS}
    fid_h = {c: np.zeros(n_bins) for c in COMPONENTS}
    fid_total = {c: 0.0 for c in COMPONENTS}
    fid_region = {c: 0.0 for c in COMPONENTS}
    selected_h2 = {c: np.zeros(n_bins) for c in COMPONENTS}

    if multipliers is None:
        multipliers = {sample: np.ones(len(frame)) for sample, frame in MC_SAMPLE_CACHE.items()}
    if stitch_scales is None:
        stitch_scales = CENTRAL_STITCH_SCALES

    for sample, frame in MC_SAMPLE_CACHE.items():
        component = SAMPLE_INFO[sample]["component"]
        idx, valid, _ = assign_bins(frame, spec)
        mult = multipliers[sample] if bootstrap_hist else np.ones(len(frame), dtype=float)
        selected = (
            frame["mc_weight_xsec_only"].to_numpy(float)
            * float(stitch_scales.get(sample, 1.0))
            * mult
            / (LUMI_RUN2_PB_INV*BRANCHING_PRODUCT)
        )
        eff = evaluate_efficiency_product(
            frame,
            EFFICIENCY_SOURCE_FOR_COMPONENT[component],
            z_replica=z_replica,
        )
        fid = selected/eff
        ii = idx[valid]
        sw = selected[valid]
        fw = fid[valid]
        selected_h[component] += np.bincount(ii, weights=sw, minlength=n_bins)
        selected_h2[component] += np.bincount(ii, weights=sw**2, minlength=n_bins)
        fid_h[component] += np.bincount(ii, weights=fw, minlength=n_bins)
        fid_total[component] += float(np.sum(fw))
        regmask = (
            valid
            & (frame["delta_y"].to_numpy(float) >= region["dy_min"])
            & (frame["delta_phi"].to_numpy(float) >= region["dphi_min"])
            & (frame["delta_phi"].to_numpy(float) <= region["dphi_max"])
        )
        fid_region[component] += float(np.sum(fid[regmask]))

    states = {}
    for component in COMPONENTS:
        response, fid_shape, total = _component_from_histograms(
            selected_h[component], fid_h[component], fid_total[component]
        )
        neff = safe_divide(selected_h[component]**2, selected_h2[component], default=0.0)
        states[component] = {
            "response": response,
            "fid_shape": fid_shape,
            "fid_total": total,
            "region_fraction": float(fid_region[component]/total),
            "neff": neff,
        }
    return states


def build_profile_problem(name):
    spec = make_spec(name)
    data_idx, data_valid, n_bins = assign_bins(data_sw, spec)
    data_weight = data_selected_contributions(data_sw)
    data_hist = np.bincount(
        data_idx[data_valid], weights=data_weight[data_valid], minlength=n_bins
    ).astype(float)
    data_var = np.bincount(
        data_idx[data_valid], weights=data_weight[data_valid]**2, minlength=n_bins
    ).astype(float)
    data_total = float(np.sum(data_hist))
    if data_total <= 0.0:
        raise ValueError("Non-positive sWeighted data integral")
    data_shape = data_hist/data_total
    J = np.eye(n_bins)/data_total - data_hist[:, None]/data_total**2
    data_cov = J @ np.diag(data_var) @ J.T
    H = helmert_contrast(n_bins)
    problem = {
        "name": name,
        "spec": spec,
        "n_bins": n_bins,
        "H": H,
        "data_idx": data_idx,
        "data_valid": data_valid,
        "data_weight": data_weight,
        "data_hist": data_hist,
        "data_var": data_var,
        "data_shape": data_shape,
        "data_cov": 0.5*(data_cov+data_cov.T),
        "data_reduced": H @ data_shape,
    }
    problem["central_states"] = _build_component_states(problem)
    problem["min_neff"] = min(
        float(np.min(problem["central_states"][c]["neff"])) for c in COMPONENTS
    )
    return problem


def build_replica_bank(problem, n, mode="joint", seed=RANDOM_SEED+100):
    if n <= 0:
        return None
    rng = np.random.default_rng(seed)
    R = np.zeros((n, len(COMPONENTS), problem["n_bins"]), dtype=float)
    F = np.zeros_like(R)
    T = np.zeros((n, len(COMPONENTS)), dtype=float)
    G = np.zeros((n, len(COMPONENTS)), dtype=float)
    ok = np.zeros(n, dtype=bool)
    z_success = []

    iterator = range(n)
    if tqdm is not None:
        iterator = tqdm(iterator, desc=f"stat replicas {problem['name']}:{mode}")

    for ir in iterator:
        try:
            # One event bootstrap is shared by template, composition and stitching.
            draw_mult = mode in {"joint", "mc_template", "stitching"}
            multipliers = {
                sample: rng.poisson(1.0, size=len(frame)).astype(float)
                if draw_mult else np.ones(len(frame), dtype=float)
                for sample, frame in MC_SAMPLE_CACHE.items()
            }
            fluctuate_stitch = mode in {"joint", "stitching"}
            stitch_scales = (
                derive_replica_stitch_scales(multipliers)
                if fluctuate_stitch else CENTRAL_STITCH_SCALES
            )
            fluctuate_eff = mode in {"joint", "efficiency"}
            z_replica = draw_efficiency_z(rng) if fluctuate_eff else None
            bootstrap_hist = mode in {"joint", "mc_template"}
            states = _build_component_states(
                problem,
                multipliers=multipliers,
                stitch_scales=stitch_scales,
                z_replica=z_replica,
                bootstrap_hist=bootstrap_hist,
            )
            for component, ic in COMPONENT_INDEX.items():
                R[ir, ic] = states[component]["response"]
                F[ir, ic] = states[component]["fid_shape"]
                T[ir, ic] = states[component]["fid_total"]
                G[ir, ic] = states[component]["region_fraction"]
            ok[ir] = True
            z_success.append(z_replica)
        except Exception as exc:
            if ir < 5:
                warnings.warn(f"Replica {ir} failed ({mode}): {exc}")

    R, F, T, G = R[ok], F[ok], T[ok], G[ok]
    required = max(
        MIN_SUCCESSFUL_REPLICAS,
        int(math.ceil(MIN_MODEL_REPLICA_SUCCESS_FRACTION * int(n))),
    )
    if len(R) < required:
        raise RuntimeError(
            f"Only {len(R)}/{int(n)} successful {mode} replicas for "
            f"{problem['name']}; need at least {required}."
        )
    return {"response": R, "fid_shape": F, "fid_total": T, "region_fraction": G,
            "z_replica": z_success if mode in {"joint", "efficiency"} else [],
            "n": len(R), "mode": mode}


def _derived_composition(fid_total):
    arr = np.asarray(fid_total, dtype=float)
    squeeze = arr.ndim == 1
    if squeeze:
        arr = arr[None, :]
    ic_scc = COMPONENT_INDEX["SPS-ccbar"]
    ic_sbb = COMPONENT_INDEX["SPS-bbbar"]
    ic_dcc = COMPONENT_INDEX["DPS-ccbar"]
    ic_dbb = COMPONENT_INDEX["DPS-bbbar"]
    fcc_sps = safe_divide(
        arr[:, ic_scc], arr[:, ic_scc] + arr[:, ic_sbb], default=0.5
    )
    fcc_dps = safe_divide(
        arr[:, ic_dcc], arr[:, ic_dcc] + arr[:, ic_dbb], default=0.5
    )
    if squeeze:
        return float(fcc_sps[0]), float(fcc_dps[0])
    return fcc_sps, fcc_dps


def mixture_from_arrays(
    response, fid_shape, fid_total, f_dps,
    f_cc_sps=None, f_cc_dps=None,
):
    """Build SPS/DPS mixture in fiducial-fraction parameterization.

    In 1-parameter mode the ccbar fractions are inferred from the fiducial MC
    totals separately for every finite-simulation replica, reproducing the
    current FINAL behavior.

    In 3-parameter mode explicit f_cc_sps and f_cc_dps values are supplied by
    the fit. They are then held at the tested parameter point while the
    component response shapes fluctuate across finite-simulation replicas.
    """
    f = float(f_dps)
    ic_scc = COMPONENT_INDEX["SPS-ccbar"]
    ic_sbb = COMPONENT_INDEX["SPS-bbbar"]
    ic_dcc = COMPONENT_INDEX["DPS-ccbar"]
    ic_dbb = COMPONENT_INDEX["DPS-bbbar"]

    response = np.asarray(response, dtype=float)
    fid_shape = np.asarray(fid_shape, dtype=float)
    fid_total = np.asarray(fid_total, dtype=float)

    squeeze = response.ndim == 2
    if squeeze:
        response = response[None, ...]
        fid_shape = fid_shape[None, ...]
        fid_total = fid_total[None, ...]

    nrep = response.shape[0]
    derived_sps, derived_dps = _derived_composition(fid_total)

    if f_cc_sps is None:
        fcc_sps = np.asarray(derived_sps, dtype=float)
    else:
        fcc_sps = np.full(nrep, float(f_cc_sps), dtype=float)

    if f_cc_dps is None:
        fcc_dps = np.asarray(derived_dps, dtype=float)
    else:
        fcc_dps = np.full(nrep, float(f_cc_dps), dtype=float)

    if np.any((fcc_sps <= 0.0) | (fcc_sps >= 1.0)):
        raise ValueError("f_cc_sps outside physical interval")
    if np.any((fcc_dps <= 0.0) | (fcc_dps >= 1.0)):
        raise ValueError("f_cc_dps outside physical interval")

    r_sps = fcc_sps[:, None]*response[:, ic_scc] + (1.0-fcc_sps)[:, None]*response[:, ic_sbb]
    r_dps = fcc_dps[:, None]*response[:, ic_dcc] + (1.0-fcc_dps)[:, None]*response[:, ic_dbb]
    t_sps = fcc_sps[:, None]*fid_shape[:, ic_scc] + (1.0-fcc_sps)[:, None]*fid_shape[:, ic_sbb]
    t_dps = fcc_dps[:, None]*fid_shape[:, ic_dcc] + (1.0-fcc_dps)[:, None]*fid_shape[:, ic_dbb]

    unnorm = (1.0-f)*r_sps + f*r_dps
    shape = safe_divide(unnorm, np.sum(unnorm, axis=1)[:, None], default=np.nan)
    t_sps = safe_divide(t_sps, np.sum(t_sps, axis=1)[:, None], default=np.nan)
    t_dps = safe_divide(t_dps, np.sum(t_dps, axis=1)[:, None], default=np.nan)

    out = {
        "shape": shape,
        "sps_response": r_sps,
        "dps_response": r_dps,
        "sps_fid_shape": t_sps,
        "dps_fid_shape": t_dps,
        "f_cc_sps": fcc_sps,
        "f_cc_dps": fcc_dps,
    }
    if squeeze:
        return {
            "shape": out["shape"][0],
            "sps_response": out["sps_response"][0],
            "dps_response": out["dps_response"][0],
            "sps_fid_shape": out["sps_fid_shape"][0],
            "dps_fid_shape": out["dps_fid_shape"][0],
            "f_cc_sps": float(out["f_cc_sps"][0]),
            "f_cc_dps": float(out["f_cc_dps"][0]),
        }
    return out


def central_component_arrays(problem):
    response = np.stack([problem["central_states"][c]["response"] for c in COMPONENTS])
    fid_shape = np.stack([problem["central_states"][c]["fid_shape"] for c in COMPONENTS])
    fid_total = np.array([problem["central_states"][c]["fid_total"] for c in COMPONENTS])
    return response, fid_shape, fid_total


def fixed_composition_for_problem(problem):
    _, _, fid_total = central_component_arrays(problem)
    fcc_sps, fcc_dps = _derived_composition(fid_total)
    return {"f_cc_sps": float(fcc_sps), "f_cc_dps": float(fcc_dps)}


def cc_fraction_constraint_penalty(problem, f_cc_sps, f_cc_dps):
    """
    Gaussian constraint contribution to -2 log L.

    The parameters remain free. The constraint only penalizes departures
    from the central fiducial MC composition.
    """
    if not (FREE_CC_FRACTIONS and USE_CC_FRACTION_CONSTRAINTS):
        return 0.0

    nominal = fixed_composition_for_problem(problem)

    pull_sps = (
        float(f_cc_sps) - nominal["f_cc_sps"]
    ) / float(CC_SPS_CONSTRAINT_SIGMA)

    pull_dps = (
        float(f_cc_dps) - nominal["f_cc_dps"]
    ) / float(CC_DPS_CONSTRAINT_SIGMA)

    return float(pull_sps**2 + pull_dps**2)


def starting_parameters(problem, start=None):
    composition = fixed_composition_for_problem(problem)
    values = {
        "f_dps": 0.35,
        "f_cc_sps": composition["f_cc_sps"],
        "f_cc_dps": composition["f_cc_dps"],
    }
    if start:
        for key in values:
            if key in start and np.isfinite(start[key]):
                values[key] = float(start[key])
    return values


def active_parameter_names():
    return ["f_dps", "f_cc_sps", "f_cc_dps"] if FREE_CC_FRACTIONS else ["f_dps"]


def mixture_parameter_kwargs(params):
    if FREE_CC_FRACTIONS:
        return {
            "f_cc_sps": float(params["f_cc_sps"]),
            "f_cc_dps": float(params["f_cc_dps"]),
        }
    return {"f_cc_sps": None, "f_cc_dps": None}


def central_mixture(problem, params):
    if np.isscalar(params):
        params = {"f_dps": float(params)}
    response, fid_shape, fid_total = central_component_arrays(problem)
    kwargs = mixture_parameter_kwargs(params)
    return mixture_from_arrays(
        response, fid_shape, fid_total, float(params["f_dps"]), **kwargs
    )


def bank_mixture(problem, params, bank_key="joint"):
    if np.isscalar(params):
        params = {"f_dps": float(params)}
    bank = problem["stat_banks"][bank_key]
    kwargs = mixture_parameter_kwargs(params)
    return mixture_from_arrays(
        bank["response"], bank["fid_shape"], bank["fid_total"],
        float(params["f_dps"]), **kwargs,
    )


def sample_covariance(arr, n_dim):
    arr = np.asarray(arr, dtype=float)
    if arr.ndim != 2 or len(arr) < 2:
        return np.zeros((n_dim, n_dim), dtype=float)
    cov = np.cov(arr, rowvar=False, ddof=1)
    cov = np.atleast_2d(cov).astype(float)
    return 0.5*(cov+cov.T)


PROBLEMS = {NOMINAL_BINNING: build_profile_problem(NOMINAL_BINNING)}
if RUN_ALTERNATE_BINNING_CROSSCHECK:
    PROBLEMS[ALTERNATE_BINNING] = build_profile_problem(ALTERNATE_BINNING)

for i, problem in enumerate(PROBLEMS.values()):
    problem["stat_banks"] = {}
    problem["stat_banks"]["joint"] = build_replica_bank(
        problem, N_MODEL_STAT_REPLICAS, mode="joint", seed=RANDOM_SEED+1000+50*i
    )
    # Dedicated systematics notebook: the central sensitivity variations
    # reuse the nominal joint statistical covariance.  The source-isolation
    # banks are intentionally not rebuilt here.

quality_rows=[]
for name, problem in PROBLEMS.items():
    row={"binning":name,"n_bins":problem["n_bins"],"min_Neff":problem["min_neff"]}
    for c in COMPONENTS:
        row[f"min_Neff_{c}"] = float(np.min(problem["central_states"][c]["neff"]))
    row["joint_stat_replicas"] = int(problem["stat_banks"]["joint"]["n"])
    row["template_stats_reportable"] = bool(problem["min_neff"] >= MIN_TEMPLATE_NEFF_REPORTABLE)
    fixed = fixed_composition_for_problem(problem)
    row["fixed_f_cc_sps"] = fixed["f_cc_sps"]
    row["fixed_f_cc_dps"] = fixed["f_cc_dps"]
    quality_rows.append(row)
BINNING_QUALITY = pd.DataFrame(quality_rows)
for _, row in BINNING_QUALITY.iterrows():
    print(
        f"[{row['binning']}] template bins={int(row['n_bins'])}, "
        f"minimum N_eff={row['min_Neff']:.3f}, "
        f"joint replicas={int(row['joint_stat_replicas'])}, "
        f"fixed f_cc(SPS)={row['fixed_f_cc_sps']:.4f}, "
        f"fixed f_cc(DPS)={row['fixed_f_cc_dps']:.4f}"
    )
    
BINNING_QUALITY.to_csv(OUTDIR/"binning_template_statistics.csv", index=False)


## Existing sPlot-variation hook

In [ ]:
def _sweight_variant_files():
    if not SPLOT_VARIATION_DIR.is_dir():
        return []
    return sorted([p for p in SPLOT_VARIATION_DIR.rglob("*.root") if p.is_file()])

def build_splot_systematic(problem):
    files = _sweight_variant_files()
    shapes=[]
    hists=[]
    weights=[]
    records=[]
    for path in files:
        try:
            frame, _ = read_logical_dataframe(path, ["sweight"])
            w = frame["sweight"].to_numpy(float)
            if len(w) != len(data_sw):
                raise ValueError(f"length {len(w)} != {len(data_sw)}")
            idx = problem["data_idx"]
            valid = problem["data_valid"]
            wpb = w/(LUMI_RUN2_PB_INV*BRANCHING_PRODUCT)
            hist = np.bincount(idx[valid], weights=wpb[valid], minlength=problem["n_bins"]).astype(float)
            total = float(np.sum(hist))
            if total <= 0.0:
                raise ValueError("non-positive weighted integral")
            shape = hist/total
            shapes.append(shape)
            hists.append(hist)
            weights.append(w.copy())
            records.append({"file":str(path),"status":"loaded"})
        except Exception as exc:
            records.append({"file":str(path),"status":f"failed: {exc}"})
    if not shapes:
        return {
            "cov": np.zeros((problem["n_bins"], problem["n_bins"])),
            "shapes": np.empty((0, problem["n_bins"])),
            "hists": np.empty((0, problem["n_bins"])),
            "weights": np.empty((0, len(data_sw))),
            "records": records,
            "available": False,
        }
    delta = np.asarray(shapes)-problem["data_shape"]
    # RMS covariance of the approved fit-variant ensemble around nominal.
    cov = delta.T @ delta / float(len(delta))
    cov = 0.5*(cov+cov.T)
    return {
        "cov": cov,
        "shapes": np.asarray(shapes),
        "hists": np.asarray(hists),
        "weights": np.asarray(weights),
        "records": records,
        "available": True,
    }


for name, problem in PROBLEMS.items():
    problem["splot_systematic"] = build_splot_systematic(problem)

SPLOT_VARIATION_AUDIT = pd.DataFrame(
    PROBLEMS[NOMINAL_BINNING]["splot_systematic"]["records"]
)
if len(SPLOT_VARIATION_AUDIT):
    print(f"[sPlot syst] {sum(SPLOT_VARIATION_AUDIT['status'].eq('loaded'))} approved variation files loaded")
SPLOT_VARIATION_AUDIT.to_csv(OUTDIR/"splot_variant_audit.csv", index=False)

# Double-counting guard for the absolute upper-panel reference band.
DIFFERENTIAL_AN143_SOURCES = set()
if PROBLEMS[NOMINAL_BINNING]["splot_systematic"]["available"]:
    DIFFERENTIAL_AN143_SOURCES.add("signal_yield")
AN143_COMMON_REL_AVAILABLE = math.sqrt(sum(
    AN143_REFERENCE_RELATIVE_SYSTEMATICS[name]**2
    for name in AN143_COMMON_NORMALIZATION_SOURCES
    if name not in DIFFERENTIAL_AN143_SOURCES
))
AN143_COMMON_REL_EFFECTIVE = (
    AN143_COMMON_REL_AVAILABLE if INCLUDE_AN143_INTEGRATED_SYSTEMATICS else 0.0
)
print("AN143 available common normalization reference =", f"{100*AN143_COMMON_REL_AVAILABLE:.3f}%")
print("AN143 common normalization used in spectra     =", f"{100*AN143_COMMON_REL_EFFECTIVE:.3f}%")
print("AN143 sources removed because propagated differentially =", sorted(DIFFERENTIAL_AN143_SOURCES))


## Nominal likelihood fit

In [ ]:
PARAMETER_NAMES = active_parameter_names()


def model_stat_covariance(problem, params, bank_key="joint"):
    bank_model = bank_mixture(problem, params, bank_key=bank_key)
    return sample_covariance(bank_model["shape"], problem["n_bins"])


def model_inadequacy_shape_covariance(shape, relative_floor):
    """Normalized-shape covariance from one global relative 2D floor."""
    p = np.asarray(shape, dtype=float)
    p = p / float(np.sum(p))
    A = np.diag(p) - np.outer(p, p)
    cov = float(relative_floor)**2 * (A @ A.T)
    cov = 0.5 * (cov + cov.T)
    if not np.allclose(np.sum(cov, axis=0), 0.0, atol=1e-12, rtol=0.0):
        raise AssertionError("model-inadequacy covariance contains a normalization mode")
    if float(np.min(np.linalg.eigvalsh(cov))) < -1e-12:
        raise AssertionError("model-inadequacy covariance is not positive semidefinite")
    return cov


def effective_model_inadequacy_rel(include_systematics, requested=None):
    if not include_systematics or not USE_MODEL_INADEQUACY:
        return 0.0
    value = MODEL_INADEQUACY_REL if requested is None else float(requested)
    return float(value)


def q_likelihood(problem, params, data_reduced=None, include_systematics=True,
                 bank_key="joint", model_inadequacy_rel=None):
    if np.isscalar(params):
        params = {"f_dps": float(params)}
    f = float(params["f_dps"])
    if not (1.0e-6 < f < 1.0-1.0e-6):
        return 1.0e30, 1.0e30, None, None, None, np.inf
    if FREE_CC_FRACTIONS:
        for key in ("f_cc_sps", "f_cc_dps"):
            if key not in params or not (1.0e-6 < float(params[key]) < 1.0-1.0e-6):
                return 1.0e30, 1.0e30, None, None, None, np.inf

    model = central_mixture(problem, params)
    H = problem["H"]
    y = problem["data_reduced"] if data_reduced is None else np.asarray(data_reduced, dtype=float)
    residual = y - H @ model["shape"]
    cov = problem["data_cov"] + model_stat_covariance(problem, params, bank_key=bank_key)
    if include_systematics:
        cov = cov + problem["splot_systematic"]["cov"]
    floor_rel = effective_model_inadequacy_rel(
        include_systematics, model_inadequacy_rel
    )
    if floor_rel > 0.0:
        cov = cov + model_inadequacy_shape_covariance(model["shape"], floor_rel)
    V = H @ cov @ H.T
    V = nearest_psd(V, 1.0e-12)
    quadratic = float(residual @ np.linalg.solve(V, residual))
    sign, logdet = np.linalg.slogdet(V)
    if sign <= 0 or not np.isfinite(logdet):
        return 1.0e30, 1.0e30, model, V, cov, np.inf
    # V depends on the physics parameters through the finite-model ensemble.
    # Therefore the Gaussian normalization term is part of the likelihood.

    constraint_penalty = 0.0

    if FREE_CC_FRACTIONS:
        constraint_penalty = cc_fraction_constraint_penalty(
            problem,
            params["f_cc_sps"],
            params["f_cc_dps"],
        )

    n2ll = float(
        quadratic
        + logdet
        + constraint_penalty
    )

    return n2ll, quadratic, model, V, cov, float(logdet)

def make_minuit(problem, include_systematics=True, data_reduced=None, start=None,
                model_inadequacy_rel=None):
    start_values = starting_parameters(problem, start=start)

    if FREE_CC_FRACTIONS:
        def objective(f_dps, f_cc_sps, f_cc_dps):
            params = {
                "f_dps": f_dps,
                "f_cc_sps": f_cc_sps,
                "f_cc_dps": f_cc_dps,
            }
            n2ll, _, _, _, _, _ = q_likelihood(
                problem, params, data_reduced=data_reduced,
                include_systematics=include_systematics,
                model_inadequacy_rel=model_inadequacy_rel,
            )
            return 0.5*n2ll

        m = Minuit(
            objective,
            f_dps=start_values["f_dps"],
            f_cc_sps=start_values["f_cc_sps"],
            f_cc_dps=start_values["f_cc_dps"],
        )
        for name in ("f_dps", "f_cc_sps", "f_cc_dps"):
            m.limits[name] = (1.0e-6, 1.0-1.0e-6)
    else:
        def objective(f_dps):
            params = {"f_dps": f_dps}
            n2ll, _, _, _, _, _ = q_likelihood(
                problem, params, data_reduced=data_reduced,
                include_systematics=include_systematics,
                model_inadequacy_rel=model_inadequacy_rel,
            )
            return 0.5*n2ll

        m = Minuit(objective, f_dps=start_values["f_dps"])
        m.limits["f_dps"] = (1.0e-6, 1.0-1.0e-6)

    m.errordef = 0.5
    m.print_level = 0
    return m


def fit_problem(problem, include_systematics=True, data_reduced=None,
                run_minos=True, start=None, model_inadequacy_rel=None):
    floor_rel = effective_model_inadequacy_rel(
        include_systematics, model_inadequacy_rel
    )
    m = make_minuit(
        problem, include_systematics, data_reduced, start=start,
        model_inadequacy_rel=floor_rel,
    )
    m.migrad()
    m.hesse()
    if run_minos:
        try:
            m.minos("f_dps")
        except Exception as exc:
            warnings.warn(f"MINOS failed for {problem['name']}: {exc}")

    values = {name: float(m.values[name]) for name in PARAMETER_NAMES}
    n2ll, quadratic, model, V, full_cov, logdet = q_likelihood(
        problem, values, data_reduced=data_reduced,
        include_systematics=include_systematics,
        model_inadequacy_rel=floor_rel,
    )

    if "f_dps" in m.merrors:
        e = m.merrors["f_dps"]
        down, up = float(-e.lower), float(e.upper)
        source = "MINOS"
    else:
        down = up = float(m.errors["f_dps"])
        source = "HESSE"

    errors = {name: float(m.errors[name]) for name in PARAMETER_NAMES}
    fixed_comp = fixed_composition_for_problem(problem)
    if not FREE_CC_FRACTIONS:
        values = {**values, **fixed_comp}
        errors = {**errors, "f_cc_sps": 0.0, "f_cc_dps": 0.0}

    return {
        "problem": problem,
        "include_systematics": include_systematics,
        "model_inadequacy_rel": float(floor_rel),
        "minuit": m,
        "valid": bool(m.valid),
        "values": values,
        "errors": errors,
        "model": model,
        "q": n2ll,
        "quadratic": quadratic,
        "logdet": logdet,
        "V_reduced": V,
        "full_cov": full_cov,
        "f_error_down": down,
        "f_error_up": up,
        "f_error_source": source,
        "data_reduced": problem["data_reduced"] if data_reduced is None else np.asarray(data_reduced,float),
    }


FITS={}
for name, problem in PROBLEMS.items():
    print("\n"+"="*76+f"\nFIT {name} ({len(PARAMETER_NAMES)} free parameter(s))\n"+"="*76)
    stat_fit = fit_problem(problem, include_systematics=False, run_minos=True)
    baseline_fit = fit_problem(
        problem, include_systematics=True, run_minos=True,
        start=stat_fit["values"], model_inadequacy_rel=0.0,
    )
    full_fit = fit_problem(
        problem, include_systematics=True, run_minos=True,
        start=baseline_fit["values"],
        model_inadequacy_rel=MODEL_INADEQUACY_REL,
    )
    FITS[name] = {"stat":stat_fit, "baseline":baseline_fit, "total":full_fit}
    print("valid:", full_fit["valid"])
    print("fDPS:", full_fit["values"]["f_dps"])
    if FREE_CC_FRACTIONS:
        print("f_cc(SPS):", full_fit["values"]["f_cc_sps"], "+/-", full_fit["errors"]["f_cc_sps"])
        print("f_cc(DPS):", full_fit["values"]["f_cc_dps"], "+/-", full_fit["errors"]["f_cc_dps"])
    else:
        print("fixed f_cc(SPS):", full_fit["values"]["f_cc_sps"])
        print("fixed f_cc(DPS):", full_fit["values"]["f_cc_dps"])
    print("statistical profile interval:", (-stat_fit["f_error_down"], +stat_fit["f_error_up"]))
    print("baseline fDPS (floor=0):    ", baseline_fit["values"]["f_dps"])
    print("full profile interval:       ", (-full_fit["f_error_down"], +full_fit["f_error_up"]))
    print("empirical shape floor:       ", f"{100*full_fit['model_inadequacy_rel']:.1f}%")
    print("differential sPlot syst input:", problem["splot_systematic"]["available"])


## Technical raw-map reconstruction closure

In [ ]:
# ================================================================
# RAW-WEIGHTED-SUM UTILITIES AND TECHNICAL MAP CLOSURE
# ================================================================

RAW_SUFFIXES = ("num_sumw", "num_sumw2", "den_sumw", "den_sumw2")
AXIS_ATOL_SYS = 1.0e-7
VALUE_ATOL_SYS = 1.0e-10


def _payload(obj):
    out = obj.to_numpy(flow=False)
    values = np.asarray(out[0], dtype=float)
    edges = tuple(np.asarray(axis, dtype=float) for axis in out[1:])
    return values, edges


def _same_axis(a, b):
    a = np.asarray(a, float)
    b = np.asarray(b, float)
    return a.shape == b.shape and np.allclose(a, b, rtol=0.0, atol=AXIS_ATOL_SYS)


def _efficiency_root_path(source, year):
    return Path(load_efficiency_source(source)[str(year)]["path"])


def load_raw_map_payload(source, year, factor):
    path = _efficiency_root_path(source, year)
    raw = {}
    raw_edges = None
    with uproot.open(path) as f:
        for suffix in RAW_SUFFIXES:
            key = f"raw/{factor}_{suffix}"
            if key not in f:
                raise KeyError(f"{path}: missing {key}")
            values, edges = _payload(f[key])
            if raw_edges is None:
                raw_edges = edges
            elif not all(_same_axis(a, b) for a, b in zip(raw_edges, edges)):
                raise RuntimeError(f"{source}/{year}/{factor}: raw axes disagree")
            raw[suffix] = values
    return raw, raw_edges


def central_from_raw(raw):
    num = np.asarray(raw["num_sumw"], float)
    den = np.asarray(raw["den_sumw"], float)
    if np.any(~np.isfinite(num)) or np.any(~np.isfinite(den)):
        raise RuntimeError("non-finite raw weighted sums")
    if np.any(den <= 0.0):
        raise RuntimeError("non-positive raw denominator")
    return num / den


def neff_from_raw(raw):
    den = np.asarray(raw["den_sumw"], float)
    den2 = np.asarray(raw["den_sumw2"], float)
    if np.any(den <= 0.0) or np.any(den2 <= 0.0):
        raise RuntimeError("invalid raw denominator statistics")
    return den**2 / den2


closure_rows = []
for source in sorted(set(EFFICIENCY_SOURCE_FOR_COMPONENT.values())):
    years = load_efficiency_source(source)
    for year in EFFICIENCY_YEARS:
        for factor in EFFICIENCY_FACTORS_USED:
            raw, raw_edges = load_raw_map_payload(source, year, factor)
            rebuilt = central_from_raw(raw)
            stored = years[year]["maps"][factor]["central"]
            if rebuilt.shape != stored.values.shape:
                raise RuntimeError(
                    f"{source}/{year}/{factor}: raw/stored shape mismatch "
                    f"{rebuilt.shape} vs {stored.values.shape}"
                )
            max_abs = float(np.max(np.abs(rebuilt - stored.values)))
            scale = np.maximum(np.abs(stored.values), 1.0e-15)
            max_rel = float(np.max(np.abs(rebuilt - stored.values) / scale))
            closure_rows.append({
                "source": source,
                "year": year,
                "factor": factor,
                "max_abs_diff": max_abs,
                "max_rel_diff": max_rel,
                "min_neff_raw": float(np.min(neff_from_raw(raw))),
                "pass": bool(max_abs <= 1.0e-8),
            })

RAW_RECONSTRUCTION_CLOSURE = pd.DataFrame(closure_rows)
RAW_RECONSTRUCTION_CLOSURE.to_csv(
    OUTDIR / "raw_map_reconstruction_closure.csv", index=False
)

if not bool(RAW_RECONSTRUCTION_CLOSURE["pass"].all()):
    bad = RAW_RECONSTRUCTION_CLOSURE.loc[~RAW_RECONSTRUCTION_CLOSURE["pass"]]
    raise RuntimeError(
        "Raw-map reconstruction closure failed:\n" + bad.to_string(index=False)
    )

print("RAW-MAP RECONSTRUCTION CLOSURE: PASS")
print(
    "maximum absolute central-map difference =",
    f"{RAW_RECONSTRUCTION_CLOSURE['max_abs_diff'].max():.3e}",
)
print(
    "minimum raw-map N_eff =",
    f"{RAW_RECONSTRUCTION_CLOSURE['min_neff_raw'].min():.2f}",
)


## Exact map-rebinning utilities

In [ ]:
# ================================================================
# EXACT REBINNING FROM STORED RAW WEIGHTED SUMS
# ================================================================


def groups_for_target_edges(original_edges, target_edges):
    original_edges = np.asarray(original_edges, float)
    target_edges = np.asarray(target_edges, float)

    if not np.isclose(target_edges[0], original_edges[0], rtol=0.0, atol=AXIS_ATOL_SYS):
        raise RuntimeError("target lower edge does not match source")
    if not np.isclose(target_edges[-1], original_edges[-1], rtol=0.0, atol=AXIS_ATOL_SYS):
        raise RuntimeError("target upper edge does not match source")

    boundaries = []
    for edge in target_edges:
        matches = np.where(
            np.isclose(original_edges, edge, rtol=0.0, atol=AXIS_ATOL_SYS)
        )[0]
        if len(matches) != 1:
            raise RuntimeError(
                f"target edge {edge} not uniquely represented in {original_edges.tolist()}"
            )
        boundaries.append(int(matches[0]))

    groups = []
    for lo, hi in zip(boundaries[:-1], boundaries[1:]):
        if hi <= lo:
            raise RuntimeError("target edges are not strictly increasing")
        groups.append(list(range(lo, hi)))
    return groups


def rebin_raw_2d(raw, source_edges, target_x, target_y):
    gx = groups_for_target_edges(source_edges[0], target_x)
    gy = groups_for_target_edges(source_edges[1], target_y)
    out = {}
    for suffix, arr in raw.items():
        arr = np.asarray(arr, float)
        rebinned = np.zeros((len(gx), len(gy)), dtype=float)
        for ix, xs in enumerate(gx):
            for iy, ys in enumerate(gy):
                rebinned[ix, iy] = float(np.sum(arr[np.ix_(xs, ys)]))
        out[suffix] = rebinned
    return out


def build_rebinned_map(source, year, factor, target_x, target_y):
    raw, edges = load_raw_map_payload(source, year, factor)
    rebinned = rebin_raw_2d(raw, edges, target_x, target_y)
    values = central_from_raw(rebinned)
    neff = neff_from_raw(rebinned)

    if factor not in {"acc_dimu", "acc_dstar"}:
        if np.any(values < -VALUE_ATOL_SYS) or np.any(values > 1.0 + VALUE_ATOL_SYS):
            raise RuntimeError(f"{source}/{year}/{factor}: strict efficiency outside [0,1]")
    else:
        if np.any(values <= 0.0):
            raise RuntimeError(f"{source}/{year}/{factor}: non-positive response ratio")

    if np.min(neff) < MIN_EFFICIENCY_MAP_NEFF:
        raise RuntimeError(
            f"{source}/{year}/{factor}: rebinned min N_eff={np.min(neff):.3f} "
            f"< {MIN_EFFICIENCY_MAP_NEFF}"
        )

    return (
        Hist2DMap(values, np.asarray(target_x, float), np.asarray(target_y, float),
                  f"{source}/{year}/{factor}/systematic_rebin"),
        neff,
    )


def build_override_set(variation_spec):
    overrides = {}
    audit = []
    for source in sorted(set(EFFICIENCY_SOURCE_FOR_COMPONENT.values())):
        for year in EFFICIENCY_YEARS:
            for factor, target in variation_spec.items():
                target_x, target_y = target
                hmap, neff = build_rebinned_map(
                    source, year, factor, target_x, target_y
                )
                overrides[(source, year, factor)] = hmap
                audit.append({
                    "source": source,
                    "year": year,
                    "factor": factor,
                    "min_neff": float(np.min(neff)),
                    "xedges": list(map(float, target_x)),
                    "yedges": list(map(float, target_y)),
                })
    return overrides, pd.DataFrame(audit)


def make_custom_efficiency_evaluator(overrides=None, year_weights=None):
    overrides = {} if overrides is None else dict(overrides)
    weights = (
        dict(EFFICIENCY_YEAR_WEIGHTS)
        if year_weights is None
        else {str(k): float(v) for k, v in year_weights.items()}
    )

    def evaluator(frame, source, z_replica=None, association_mode=None):
        if z_replica is not None:
            raise RuntimeError(
                "Systematic central variations intentionally do not fluctuate "
                "efficiency-map statistics."
            )
        association_mode_local = (
            ASSOCIATION_EFF_MODE if association_mode is None else association_mode
        )
        factors = [
            "acc_dimu",
            "acc_dstar",
            "eff_cuts_dimu",
            "eff_cuts_dstar",
            "eff_trigger",
            "eff_asso_pt" if association_mode_local == "pt" else "eff_asso_rap",
        ]

        weighted = np.zeros(len(frame), dtype=float)
        total_weight = 0.0
        years = load_efficiency_source(source)

        for year, year_weight in weights.items():
            if year not in years:
                raise KeyError(f"Missing efficiency response year {year}")
            product = np.ones(len(frame), dtype=float)
            for factor in factors:
                info = years[year]["maps"][factor]
                hmap = overrides.get((source, year, factor), info["central"])
                x, y = factor_coordinates(frame, factor)
                product *= hmap.eval(x, y)
            weighted += year_weight * product
            total_weight += year_weight

        if total_weight <= 0.0:
            raise RuntimeError("non-positive custom response weight")
        out = weighted / total_weight
        if np.any(~np.isfinite(out)) or np.any(out <= 0.0):
            raise RuntimeError("invalid custom efficiency product")
        return np.maximum(out, EFFICIENCY_FLOOR)

    return evaluator


## Systematic refit utilities

In [ ]:
# ================================================================
# REBUILD CENTRAL PROBLEM UNDER A DETECTOR-RESPONSE VARIATION
# ================================================================

NOMINAL_PROBLEM = PROBLEMS[NOMINAL_BINNING]
NOMINAL_FIT = FITS[NOMINAL_BINNING]["total"]
NOMINAL_ACTIVE_PARAMS = {
    name: float(NOMINAL_FIT["values"][name])
    for name in PARAMETER_NAMES
}
NOMINAL_FDPS = float(NOMINAL_FIT["values"]["f_dps"])
NOMINAL_MODEL_AT_NOMINAL_PARAMS = central_mixture(
    NOMINAL_PROBLEM, NOMINAL_ACTIVE_PARAMS
)
NOMINAL_SHAPE = np.asarray(NOMINAL_MODEL_AT_NOMINAL_PARAMS["shape"], float)


def rebuild_problem_with_efficiency_evaluator(label, evaluator):
    global evaluate_efficiency_product
    original = evaluate_efficiency_product
    try:
        evaluate_efficiency_product = evaluator
        problem = build_profile_problem(NOMINAL_BINNING)
    finally:
        evaluate_efficiency_product = original

    # Systematic shifts are evaluated with the same statistical covariance
    # reference as the nominal fit.  This isolates the central detector-response
    # change instead of re-sampling MC statistics for every systematic point.
    problem["name"] = str(label)
    problem["stat_banks"] = NOMINAL_PROBLEM["stat_banks"]
    problem["splot_systematic"] = NOMINAL_PROBLEM["splot_systematic"]
    return problem


def evaluate_systematic_variation(label, evaluator, source_class):
    problem = rebuild_problem_with_efficiency_evaluator(label, evaluator)
    fit = fit_problem(
        problem,
        include_systematics=True,
        run_minos=False,
        start=NOMINAL_FIT["values"],
        model_inadequacy_rel=MODEL_INADEQUACY_REL,
    )
    if not fit["valid"]:
        raise RuntimeError(f"Systematic variation {label} produced invalid fit")

    same_physics_model = central_mixture(problem, NOMINAL_ACTIVE_PARAMS)
    shape = np.asarray(same_physics_model["shape"], float)

    return {
        "label": label,
        "source_class": source_class,
        "problem": problem,
        "fit": fit,
        "fDPS": float(fit["values"]["f_dps"]),
        "delta_fDPS": float(fit["values"]["f_dps"] - NOMINAL_FDPS),
        "shape_at_nominal_physics": shape,
        "delta_shape": shape - NOMINAL_SHAPE,
    }


print("Nominal fDPS =", NOMINAL_FDPS)


## Run-2 response-year sensitivity

In [ ]:
# ================================================================
# SYSTEMATIC 1: RESPONSE-YEAR DEPENDENCE
# ================================================================
# Temporary Run-2 proxy diagnostic while 2016APV/2016 efficiency maps are
# unavailable.  The nominal response is luminosity-weighted 2017+2018.

SYSTEMATIC_VARIATIONS = {}

YEAR_VARIATION_WEIGHTS = {
    "year_2017_only": {"2017": 1.0},
    "year_2018_only": {"2018": 1.0},
}

for label, weights in YEAR_VARIATION_WEIGHTS.items():
    evaluator = make_custom_efficiency_evaluator(year_weights=weights)
    result = evaluate_systematic_variation(label, evaluator, "year_response")
    SYSTEMATIC_VARIATIONS[label] = result
    print(
        f"{label:24s} fDPS={result['fDPS']:.6f} "
        f"delta={result['delta_fDPS']:+.6f}"
    )

YEAR_DELTA_F = np.asarray([
    SYSTEMATIC_VARIATIONS[k]["delta_fDPS"]
    for k in YEAR_VARIATION_WEIGHTS
], float)
YEAR_FDPS_ENVELOPE = float(np.max(np.abs(YEAR_DELTA_F)))
YEAR_FDPS_RMS = float(np.sqrt(np.mean(YEAR_DELTA_F**2)))

YEAR_DELTA_SHAPES = np.asarray([
    SYSTEMATIC_VARIATIONS[k]["delta_shape"]
    for k in YEAR_VARIATION_WEIGHTS
], float)
YEAR_RESPONSE_COV = (
    YEAR_DELTA_SHAPES.T @ YEAR_DELTA_SHAPES / len(YEAR_DELTA_SHAPES)
)
YEAR_RESPONSE_COV = 0.5 * (YEAR_RESPONSE_COV + YEAR_RESPONSE_COV.T)

np.save(OUTDIR / "year_response_covariance.npy", YEAR_RESPONSE_COV)
print("year-response |delta fDPS| envelope =", YEAR_FDPS_ENVELOPE)
print("year-response delta fDPS RMS        =", YEAR_FDPS_RMS)


## Efficiency-map binning stability

In [ ]:
# ================================================================
# SYSTEMATIC 2: EFFICIENCY-MAP BINNING STABILITY
# ================================================================
# Only exact coarsenings of the validated nominal maps are used.
# All alternative maps are reconstructed from stored raw sumw/sumw2 and must
# pass the same N_eff >= 25 gate.  No interpolation is used.

DIMU_Y = [0.0, 0.9, 1.2]
ASSO_JPSI_PT = [25.0, 100.0]

BINNING_VARIATION_SPECS = {
    # Common coarsening of the three dimuon-response maps.
    "dimuon_pt_25_50_100": {
        "acc_dimu": ([25.0, 50.0, 100.0], DIMU_Y),
        "eff_cuts_dimu": ([25.0, 50.0, 100.0], DIMU_Y),
        "eff_trigger": ([25.0, 50.0, 100.0], DIMU_Y),
    },
    "dimuon_pt_25_30_100": {
        "acc_dimu": ([25.0, 30.0, 100.0], DIMU_Y),
        "eff_cuts_dimu": ([25.0, 30.0, 100.0], DIMU_Y),
        "eff_trigger": ([25.0, 30.0, 100.0], DIMU_Y),
    },
    # Coarsen only the D* pT axis of the association map.  The nominal J/psi
    # pT axis is already a single [25,100] bin by the validated common-grid
    # contract.
    "association_dstar_pt_4_20_60": {
        "eff_asso_pt": (ASSO_JPSI_PT, [4.0, 20.0, 60.0]),
    },
    "association_dstar_pt_4_10_30_60": {
        "eff_asso_pt": (ASSO_JPSI_PT, [4.0, 10.0, 30.0, 60.0]),
    },
}

BINNING_AUDITS = []
for label, spec in BINNING_VARIATION_SPECS.items():
    overrides, audit = build_override_set(spec)
    audit.insert(0, "variation", label)
    BINNING_AUDITS.append(audit)

    evaluator = make_custom_efficiency_evaluator(overrides=overrides)
    result = evaluate_systematic_variation(label, evaluator, "efficiency_binning")
    SYSTEMATIC_VARIATIONS[label] = result

    print(
        f"{label:34s} fDPS={result['fDPS']:.6f} "
        f"delta={result['delta_fDPS']:+.6f}; "
        f"min N_eff={audit['min_neff'].min():.2f}"
    )

BINNING_AUDIT = pd.concat(BINNING_AUDITS, ignore_index=True)
BINNING_AUDIT.to_csv(OUTDIR / "efficiency_binning_variation_audit.csv", index=False)

BINNING_LABELS = list(BINNING_VARIATION_SPECS)
BINNING_DELTA_F = np.asarray([
    SYSTEMATIC_VARIATIONS[k]["delta_fDPS"]
    for k in BINNING_LABELS
], float)
BINNING_FDPS_ENVELOPE = float(np.max(np.abs(BINNING_DELTA_F)))
BINNING_FDPS_RMS = float(np.sqrt(np.mean(BINNING_DELTA_F**2)))

BINNING_DELTA_SHAPES = np.asarray([
    SYSTEMATIC_VARIATIONS[k]["delta_shape"]
    for k in BINNING_LABELS
], float)
BINNING_COV = (
    BINNING_DELTA_SHAPES.T @ BINNING_DELTA_SHAPES / len(BINNING_DELTA_SHAPES)
)
BINNING_COV = 0.5 * (BINNING_COV + BINNING_COV.T)

np.save(OUTDIR / "efficiency_binning_covariance.npy", BINNING_COV)
print("binning |delta fDPS| envelope =", BINNING_FDPS_ENVELOPE)
print("binning delta fDPS RMS        =", BINNING_FDPS_RMS)


## Final systematic-study outputs

In [ ]:
# ================================================================
# SUMMARY + MACHINE-READABLE OUTPUTS FOR THE FINAL NOTEBOOK
# ================================================================

variation_rows = [{
    "label": "nominal_2017p2018",
    "source_class": "nominal",
    "fDPS": NOMINAL_FDPS,
    "delta_fDPS": 0.0,
}]

for label, result in SYSTEMATIC_VARIATIONS.items():
    variation_rows.append({
        "label": label,
        "source_class": result["source_class"],
        "fDPS": result["fDPS"],
        "delta_fDPS": result["delta_fDPS"],
    })

VARIATION_SUMMARY = pd.DataFrame(variation_rows)
VARIATION_SUMMARY.to_csv(OUTDIR / "efficiency_systematic_variations.csv", index=False)

TOTAL_EFFICIENCY_RESPONSE_COV = YEAR_RESPONSE_COV + BINNING_COV
TOTAL_EFFICIENCY_RESPONSE_COV = 0.5 * (
    TOTAL_EFFICIENCY_RESPONSE_COV + TOTAL_EFFICIENCY_RESPONSE_COV.T
)
np.save(
    OUTDIR / "efficiency_response_systematic_covariance_total.npy",
    TOTAL_EFFICIENCY_RESPONSE_COV,
)

# Store the actual variation shapes as well so the final notebook can rebuild
# or inspect the covariance without relying only on the precomputed matrix.
shape_payload = {
    "nominal": NOMINAL_SHAPE,
}
for label, result in SYSTEMATIC_VARIATIONS.items():
    shape_payload[label] = result["shape_at_nominal_physics"]
np.savez(OUTDIR / "efficiency_systematic_shapes.npz", **shape_payload)

summary = {
    "analysis": "JpsiDstar 2D profile likelihood",
    "data_period": "full Run2",
    "efficiency_nominal_response": "luminosity-weighted 2017+2018",
    "nominal_fDPS": NOMINAL_FDPS,
    "year_response": {
        "temporary_until_2016_available": True,
        "delta_fDPS_envelope": YEAR_FDPS_ENVELOPE,
        "delta_fDPS_rms": YEAR_FDPS_RMS,
        "covariance_file": "year_response_covariance.npy",
    },
    "efficiency_binning": {
        "delta_fDPS_envelope": BINNING_FDPS_ENVELOPE,
        "delta_fDPS_rms": BINNING_FDPS_RMS,
        "covariance_file": "efficiency_binning_covariance.npy",
    },
    "raw_map_reconstruction_closure": {
        "status": "PASS",
        "max_abs_difference": float(RAW_RECONSTRUCTION_CLOSURE["max_abs_diff"].max()),
        "note": "technical reconstruction closure only; not an independent physics closure",
    },
    "independent_efficiency_closure": {
        "status": "PENDING_INDEPENDENT_TEST_SAMPLE",
        "assigned_uncertainty": None,
    },
    "splot_model_systematic": {
        "status": "PENDING_VARIANT_SWEIGHTS",
        "assigned_uncertainty": None,
    },
    "combined_efficiency_response_covariance_file": (
        "efficiency_response_systematic_covariance_total.npy"
    ),
}

with open(OUTDIR / "efficiency_systematics_for_final.json", "w") as f:
    json.dump(summary, f, indent=2)

print("\nSYSTEMATICS SUMMARY")
print(VARIATION_SUMMARY.to_string(index=False))
print()
print("year-response envelope     =", YEAR_FDPS_ENVELOPE)
print("efficiency-binning envelope=", BINNING_FDPS_ENVELOPE)
print(
    "quadrature of scalar envelopes =",
    math.sqrt(YEAR_FDPS_ENVELOPE**2 + BINNING_FDPS_ENVELOPE**2),
)
print()
print("Independent efficiency closure: PENDING (no number invented)")
print("sPlot model systematic:          PENDING")
print("Outputs written to:", OUTDIR)


In [ ]:
# ================================================================
# COMPACT DIAGNOSTIC PLOT
# ================================================================

plot_df = VARIATION_SUMMARY.loc[
    VARIATION_SUMMARY["source_class"] != "nominal"
].copy()

fig, ax = plt.subplots(figsize=(8.8, 5.4), layout="constrained")
y = np.arange(len(plot_df))
ax.axvline(0.0, linewidth=1.0)
ax.scatter(plot_df["delta_fDPS"], y, s=55)
ax.set_yticks(y, plot_df["label"])
ax.set_xlabel(r"$\Delta f_{\mathrm{DPS}}$ relative to nominal")
ax.set_ylabel("")
ax.set_title("Efficiency-response systematic variations")
cms_label(ax, "Internal", LUMI_RUN2_FB_INV)
save_and_show(fig, "efficiency_systematic_fdps_shifts_CMS", OUTDIR)


## Interpretation

The output intended for later integration is `efficiency_systematics_for_final.json` plus the three covariance files.  The **year-response covariance is temporary** and should be removed/replaced once 2016APV and 2016 efficiency maps are available.

The raw-sum reconstruction check is a technical closure of the stored maps; it is **not** an independent detector-efficiency closure.  No uncertainty is assigned to the independent closure until an independent truth/test sample exists.
